# **Initialization**

In [1]:
print('Start')

Start


In [2]:
import numpy as np
import math
import random
import sys
import pulp
import vrplib
import re
import os
import modified_didppy as m_dp
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.optimize import linear_sum_assignment
from numpy.linalg import eigh
import time
from docplex.mp.model import Model

# **Data**

In [48]:
def read_tsp_cappart_format(file_path):
    """
    Parses the TSP/TSPTW text files from the specified directory.
    Structure:
    - n (int)
    - n*n distance matrix entries
    - n*2 time window entries (ignored for TSP)
    - n x_coords (ignored)
    - n y_coords (ignored)
    """
    with open(file_path, 'r') as f:
        # split() handles all whitespace (newlines and spaces) automatically
        values = f.read().split()

    iterator = iter(values)
    
    try:
        # 1. Read Number of Nodes
        n = int(next(iterator))
        
        # 2. Read Distance Matrix (n x n)
        # The file contains a flattened list of integer distances
        c = []
        for i in range(n):
            row = []
            for j in range(n):
                val = float(next(iterator)) # Read as float first to be safe
                row.append(int(val))        # Convert to int as per your DIDP model type
            c.append(row)
            
        # The rest of the file (Time windows, coords) is ignored for pure TSP
        # but the iterator ensures we consumed exactly what we needed.
        
        return n, c

    except StopIteration:
        raise ValueError(f"File {file_path} ended unexpectedly.")

In [49]:
base_path = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_TSPTW_dual_bounds_and_models\n20\95.txt"

try:   
    number_of_customers, distance_list = read_tsp_cappart_format(base_path)
    print(f'Number of customer is {number_of_customers}')
    print(f'Distance matrix is {distance_list}')

except FileNotFoundError:
    print(f"Error: The file at {base_path} was not found.")


Number of customer is 20
Distance matrix is [[0, 45, 47, 44, 78, 53, 83, 14, 93, 16, 7, 28, 7, 73, 85, 78, 70, 66, 35, 35], [45, 0, 35, 37, 84, 15, 53, 55, 67, 59, 46, 20, 50, 63, 44, 78, 34, 31, 72, 64], [47, 35, 0, 4, 49, 27, 38, 48, 47, 55, 52, 24, 48, 30, 52, 44, 31, 28, 55, 44], [44, 37, 4, 0, 47, 30, 42, 45, 50, 52, 50, 24, 46, 31, 55, 42, 35, 31, 52, 40], [78, 84, 49, 47, 0, 76, 64, 69, 61, 76, 84, 69, 75, 30, 92, 11, 72, 70, 59, 48], [53, 15, 27, 30, 76, 0, 38, 61, 52, 67, 56, 25, 58, 51, 32, 69, 19, 16, 75, 66], [83, 53, 38, 42, 64, 38, 0, 86, 14, 93, 88, 57, 86, 34, 31, 54, 20, 23, 93, 80], [14, 55, 48, 45, 69, 61, 86, 0, 94, 7, 19, 36, 8, 70, 93, 71, 75, 71, 21, 23], [93, 67, 47, 50, 61, 52, 14, 94, 0, 102, 98, 68, 95, 32, 44, 50, 34, 37, 98, 85], [16, 59, 55, 52, 76, 67, 93, 7, 102, 0, 18, 41, 9, 77, 99, 78, 82, 78, 23, 28], [7, 46, 52, 50, 84, 56, 88, 19, 98, 18, 0, 32, 11, 79, 88, 84, 73, 69, 40, 41], [28, 20, 24, 24, 69, 25, 57, 36, 68, 41, 32, 0, 33, 54, 57, 66, 42, 38,

# **Utility functions**

In [52]:
# ==========================================
# 1. EVOLUTIONARY ALGORITHM HYPERPARAMETERS
# ==========================================
POPULATION_SIZE = 0        # Size of the population in each generation
GENERATIONS = 0           # Number of generations to run
MUTATION_RATE = 0         # Probability of mutating an individual
CROSSOVER_RATE = 0        # Probability of performing crossover
ELITISM_RATE = 0
# ==========================================
# 2. OPERATOR PARAMETERS
# ==========================================
# Bounds for the coefficients generated for weighted blocks (e.g., 5.5 * h1)
LB_range_of_constant = 0 
UB_range_of_constant = 0 
# Depth limits for the RPN trees (used in Ramped Half-and-Half generator)
min_chromosome_length = 0               # Minimum depth of the initial trees
max_chromosome_length = 0               # Maximum depth of the initial trees
# Probability of selecting the best individual in the  tournament selection
# Tournament size for parent selection
tournament_size=0
tournament_probability=0
# Mutation: Maximum depth allowed for the *newly generated* subtree during mutation
mutation_max_subtree_depth = random.randint(min_chromosome_length, max_chromosome_length)  # Randomly chosen between 1 and 3
# 1-Point Crossover: Probability of using Homology (matching structure) vs Random fallback
homology_1_point_crossover_probability = 0
# Subtree Crossover: Probability of swapping a Function (Branch) vs Terminal (Leaf)
subtree_crossover_probability = 0
# Uniform Crossover: Probability of swapping genes at a specific index
uniform_crossover_probability = 0
# ==========================================
# 4. OTHER PARAMETERS
# ==========================================
# The Ground Truth optimal cost for the specific problem instance
# Used to calculate fitness (deviation from optimal)
reference_point = OPTIMAL_COST_REFERENCE = 0
# Time limit (in seconds) for the DIDP solver to run per chromosome evaluation
time_limit = SOLVER_TIME_LIMIT = 0 #seconds

In [53]:
available_operations = ["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"]

def extract_chromosome_from_chromosome_fitness_dict(chromosome_fitness_dict):
    """
    Extracts the chromosome from a chromosome-fitness dictionary.
    """
    return chromosome_fitness_dict.get('chromosome')

def automatic_creation_of_dual_bounds_registry(local_scope):
    """
    Automatically builds a registry dict from the local scope.
    It selects all variables that:
    1. Start with 'h' (e.g., 'h1', 'h_unvisited')
    2. Are callable (are functions)
    """
    return {
        name: func 
        for name, func in local_scope.items() 
        if name.startswith("h") and callable(func)
    }

def is_operator(gene, available_operations = available_operations):
    return gene in available_operations

def analyzing_chromosome_based_on_rpn_structure(chromosome_fitness_dict, 
                                                available_operations = available_operations):
    """
    Scans a chromosome to identify all valid subtrees.
    Returns a dictionary categorizing them into 'TERMINALS' and 'FUNCTIONS'.
    """
    
    chromosome = extract_chromosome_from_chromosome_fitness_dict(chromosome_fitness_dict)
    # Stores tuples of (start_index, end_index)
    structure = {
        "TERMINALS": [],  # Leaves or Weighted Blocks [c, h, *]
        "FUNCTIONS": []   # Complex operators (ADD, MAX, etc)
    }
    
    # We scan backwards (Right to Left) because RPN roots are on the right
    i = len(chromosome) - 1
    
    while i >= 0:
        root_gene = chromosome[i]
        end_index = i
        
        # Logic to find the start_index of the subtree
        required_inputs = 1 # The root needs to be produced
        current_pos = i
        
        while required_inputs > 0:
            gene = chromosome[current_pos]
            # If the sub tree is an Operator, it needs 2 input, so we produce 1 more requirement (Net +1 dependency)
            if is_operator(gene, available_operations):
                required_inputs += 1 
            else:
                required_inputs -= 1 # Terminal satisfies only 1 dependent component
            current_pos -= 1
        start_index = current_pos + 1
        
        # --- CATEGORIZATION LOGIC ---
        subtree_slice = chromosome[start_index : end_index+1]
        
        # Check if it is a "Weighted Terminal Block" [float, str, MULTIPLY]
        # The prompt specifically asks to treat these as terminals
        is_weighted_block = (
            len(subtree_slice) == 3 and 
            isinstance(subtree_slice[0], (int, float)) and
            isinstance(subtree_slice[1], str) and 
            subtree_slice[2] == "MULTIPLY"
        )
        
        # Check if it is a raw terminal (single item)
        is_raw_terminal = (start_index == end_index)
        
        if is_weighted_block or is_raw_terminal:
            structure["TERMINALS"].append((start_index, end_index))
        else:
            structure["FUNCTIONS"].append((start_index, end_index))
            
        # Move to the next node to the left
        i -= 1
        
        # Optimization: The loop above `while i >= 0` scans every index.
        # The inner logic finds the subtree rooted at `i`.
        # Since every index in a valid RPN is the root of *some* subtree (even if just itself),
        # we let the loop continue naturally.

    return structure

def generate_random_terminal_block(dual_bound_functions_dict, 
                                LB_range_of_constant = LB_range_of_constant, 
                                UB_range_of_constant = UB_range_of_constant
                                ):
    """
    Creates a single terminal unit (leaf) for the RPN list.
    It randomly decides whether to wrap the heuristic with a coefficient.
    """
    keys = list(dual_bound_functions_dict.keys())
    h_name = random.choice(keys)
    if random.random() < 0.5:
        # Weighted: [coef, h, "MULTIPLY"]
        coef = round(random.uniform(LB_range_of_constant, UB_range_of_constant), 2)
        return [coef, h_name, "MULTIPLY"]
    else:
        # Raw: [h]
        return [h_name]

def generate_rpn_tree_recursive(current_depth, max_node_depth, method, 
                                dual_bound_functions_dict,
                                LB_range_of_constant = LB_range_of_constant, 
                                UB_range_of_constant = UB_range_of_constant, 
                                available_operations = available_operations):
    """
    Recursively builds an RPN list using standard GP growth logic.
    """
    # --- BASE CASE: Hit Depth Limit ---
    if current_depth >= max_node_depth:
        # Must return a terminal
        return generate_random_terminal_block(dual_bound_functions_dict = dual_bound_functions_dict, 
                                            LB_range_of_constant = LB_range_of_constant, 
                                            UB_range_of_constant = UB_range_of_constant
                                            )

    # --- SELECTION: Choose between Function or Terminal ---
    if method == "FULL":
        # FULL: Always branch until max_depth is hit
        choice = "FUNCTION"
    else:
        # GROW: Randomly pick Function or Terminal
        # (Standard GP often uses a probability here, e.g., based on set sizes)
        # Here we use 50/50 for simplicity, or you can weight it.
        choice = random.choice(["FUNCTION", "TERMINAL"])

    # --- CONSTRUCTION ---
    if choice == "TERMINAL":
        return generate_random_terminal_block(dual_bound_functions_dict = dual_bound_functions_dict, 
                                            LB_range_of_constant = LB_range_of_constant,
                                            UB_range_of_constant = UB_range_of_constant
                                            )

    else: # FUNCTION (Internal Node)
        op = random.choice(available_operations)

        # Recursively generate left and right branches
        # Note: Binary operators always need 2 children
        left_rpn = generate_rpn_tree_recursive(
            current_depth=current_depth + 1, 
            max_node_depth=max_node_depth, 
            method = method, 
            dual_bound_functions_dict = dual_bound_functions_dict, 
            LB_range_of_constant=LB_range_of_constant,
            UB_range_of_constant=UB_range_of_constant, 
            available_operations=available_operations
        )
        
        right_rpn = generate_rpn_tree_recursive(
            current_depth=current_depth + 1, 
            max_node_depth=max_node_depth, 
            method=method, 
            dual_bound_functions_dict=dual_bound_functions_dict, 
            LB_range_of_constant=LB_range_of_constant,
            UB_range_of_constant=UB_range_of_constant, 
            available_operations=available_operations
        )

        # Combine in RPN order: Left, Right, Operator
        return left_rpn + right_rpn + [op]

def get_rpn_node_arity(gene, 
                    available_operations = available_operations):
    """
    Returns 2 for binary operators, 0 for terminals.
    Includes a check to ensure 'gene' is a string before checking the list.
    """
    # isinstance is a built-in Python function, it should always be available.
    # We check if it is a string to avoid errors if 'gene' is a number (float/int).
    if isinstance(gene, str) and gene in available_operations:
        return 2
    return 0

# **1. Model and dual bound declaration**

In [58]:
def creation_of_didp_model_function():
    """
    Creates the CVRP DIDP model and returns it along with necessary metadata 
    for the heuristic functions.
    """
    n = number_of_customers
    c = distance_list 
    
    # 2. Initialize Model
    # Note: Ensure float_cost matches your data. Your snippet used False (Int), 
    # so we explicitly cast distances to Int in the reader.
    model = m_dp.Model(maximize=False, float_cost=False)

    customer = model.add_object_type(number=n)

    # 3. State Variables
    # U: Unvisited set (excluding depot 0)
    unvisited = model.add_set_var(object_type=customer, target=list(range(1, n)))
    # i: Current location
    location = model.add_element_var(object_type=customer, target=0)

    # 4. Resource Tables
    travel_time = model.add_int_table(c)

    # 5. Transitions
    # Visit customer j
    for j in range(1, n):
        visit = m_dp.Transition(
            name="visit {}".format(j),
            cost=travel_time[location, j] + m_dp.IntExpr.state_cost(),
            preconditions=[unvisited.contains(j)],
            effects=[
                (unvisited, unvisited.remove(j)),
                (location, j),
            ],
        )
        model.add_transition(visit)

    # Return to depot
    # Note: Removed 'time' effect from your snippet as it wasn't defined in the variables
    return_to_depot = m_dp.Transition(
        name="return",
        cost=travel_time[location, 0] + m_dp.IntExpr.state_cost(),
        effects=[
            (location, 0),
        ],
        preconditions=[unvisited.is_empty(), location != 0],
    )
    model.add_transition(return_to_depot)

    # 6. Base Case
    model.add_base_case([unvisited.is_empty(), location == 0])

    # 8. Create Bundle (Model + Metadata)
    # This metadata dict allows your heuristics (like MST or assignment) 
    # to access the raw matrix data later.
    metadata = {
        "num_nodes": n,
        "distance_matrix": c,
        "unvisited_var": unvisited,
        "location_var": location,
        # Add other keys if your dual bounds need them
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

In [59]:
def create_persistent_lp_relaxation_dual_bounds(metadata):
    # --- Extract Static Data ---
    n_nodes = metadata['num_nodes']
    unvisited_set_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    dist_matrix = metadata['distance_matrix']
    
    # ==========================================
    # 1. INITIALIZATION (Runs Once)
    # ==========================================
    # Create the model instance only ONCE
    mdl = Model(name='Persistent_TSP_Relaxation')
    
    # Optimization Parameters for Speed
    mdl.parameters.threads = 1
    mdl.parameters.lpmethod = 1  # Primal Simplex is efficient for re-optimization
    mdl.log_output = False       # Silence output

    # --- Create Variables ---
    # x[i, j]: Flow variables (Continuous 0-1)
    x = {}
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                x[(i, j)] = mdl.continuous_var(lb=0, ub=1, name=f'x_{i}_{j}')

    # u[i]: MTZ potential variables
    u = {i: mdl.continuous_var(lb=0, ub=n_nodes, name=f'u_{i}') for i in range(n_nodes)}

    # --- Create Constraints (Store references to update them later) ---
    # We create constraints for ALL nodes initially.
    # We will toggle their RHS (Right Hand Side) between 1 and 0 dynamically.
    
    cons_out = {} # Constraint: Sum(x_ij) = RHS
    cons_in = {}  # Constraint: Sum(x_ji) = RHS
    
    for i in range(n_nodes):
        # Outgoing flow
        # sum(x[i, j] for all j) == RHS
        expr_out = mdl.sum(x[(i, j)] for j in range(n_nodes) if i != j)
        cons_out[i] = mdl.add_constraint(expr_out == 1, ctname=f'deg_out_{i}')
        
        # Incoming flow
        # sum(x[j, i] for all j) == RHS
        expr_in = mdl.sum(x[(j, i)] for j in range(n_nodes) if i != j)
        cons_in[i] = mdl.add_constraint(expr_in == 1, ctname=f'deg_in_{i}')

    # MTZ Constraints (Static - they rely on x and u values)
    # u[i] - u[j] + N * x[i,j] <= N - 1
    # We don't need to remove these; if x[i,j] is forced to 0, the constraint becomes loose (valid).
    for i in range(n_nodes):
        if i == 0: continue
        for j in range(n_nodes):
            if j == 0 or i == j: continue
            mdl.add_constraint(
                u[i] - u[j] + n_nodes * x[(i, j)] <= n_nodes - 1
            )

    # --- Static Objective ---
    obj_expr = mdl.sum(dist_matrix[i][j] * x[(i, j)] 
                    for i in range(n_nodes)
                    for j in range(n_nodes) if i != j)
    mdl.minimize(obj_expr)

    # ==========================================
    # 2. DYNAMIC HEURISTIC (Runs many times)
    # ==========================================
    def h_lp_relaxation(state):
        # A. Identify Active Nodes
        # The 'Active Subgraph' consists of: Current Node + Unvisited Nodes + Depot
        unvisited = state[unvisited_set_var]
        current_node = state[location_var]
        
        # Quick exit for solved state
        if not unvisited and current_node == 0: 
            return 0.0

        # Construct a fast lookup set for active nodes
        # 0 (Depot) is always part of the formulation in this relaxation
        active_set = set(unvisited)
        active_set.add(current_node)
        active_set.add(0) 

        # B. Update Model (The Optimization)
        # Instead of rebuilding, we just switch the "power" on/off for nodes
        
        for i in range(n_nodes):
            if i in active_set:
                # ACTIVE NODE: Must have degree 1 (Flow = 1)
                cons_out[i].rhs = 1
                cons_in[i].rhs = 1
                # Ensure u-variable is active (allowed to be > 0)
                u[i].ub = n_nodes
            else:
                # INACTIVE NODE: Must have degree 0 (Flow = 0)
                # Setting RHS to 0 forces all connected x_ij variables to 0
                # because x_ij >= 0. This effectively removes the node.
                cons_out[i].rhs = 0
                cons_in[i].rhs = 0
                # Fix u-variable to 0 to help solver
                u[i].ub = 0

        # C. Solve Re-optimized Model
        # cplex/docplex is smart enough to use the previous basis for speed
        sol = mdl.solve()
        
        if sol:
            return float(sol.objective_value)
        return 0.0

    return h_lp_relaxation

def dual_bound_expression_function(didp_bundle):
    " Returns a dictionary of heuristic functions (bounds) bound to the model data."
    
    model, metadata = didp_bundle
    
    # Extract metadata
    distance_list = metadata['distance_matrix']
    cost_matrix = np.array(distance_list) # Numpy version for calculations
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    num_nodes = metadata['num_nodes'] # Assumed available from creation function

    # --- Pre-computation for h_global_min_flow (Bound 2) ---
    # We calculate the global min outgoing and incoming edges for every node once.
    # This matches the 'min_to' and 'min_from' tables in the DIDP snippet.
    
    # masked_cost: diagonal is infinity to ignore self-loops
    masked_cost = cost_matrix.astype(float).copy()
    np.fill_diagonal(masked_cost, np.inf)
    
    # min_outgoing[i] = min cost to leave node i
    min_outgoing_arr = np.min(masked_cost, axis=1)
    
    # min_incoming[j] = min cost to enter node j
    min_incoming_arr = np.min(masked_cost, axis=0)

    # ==========================================
    # 1. Degree Average Bound (Local Subgraph) [NEW]
    # ==========================================
    def h_degree_average(state):
        U = state[unvisited_var]
        curr = state[location_var]
        
        # If no unvisited nodes and we are at depot (0), cost is 0
        if not U and curr == 0:
            return 0.0

        # Define active nodes for the path: Current -> [Unvisited] -> Depot (0)
        # We need to construct the submatrix for these specific nodes
        active_nodes = [curr] + sorted(list(U))
        if 0 not in active_nodes:
            active_nodes.append(0)
            
        # Extract submatrix
        sub_mat = cost_matrix[np.ix_(active_nodes, active_nodes)].astype(float)
        np.fill_diagonal(sub_mat, np.inf)

        # Calculate mins within this specific subgraph
        # axis=0 is min down columns (Incoming), axis=1 is min across rows (Outgoing)
        mins_in = np.min(sub_mat, axis=0) 
        mins_out = np.min(sub_mat, axis=1)
        
        # Logic for Path Constraints:
        # 1. Current Node (index 0 in active_nodes): Needs Outgoing, but NO Incoming
        # 2. Depot Node (index -1 in active_nodes): Needs Incoming, but NO Outgoing
        # 3. Intermediate (Unvisited): Need BOTH
        
        # Sum of valid Incoming edges (Everyone except Current)
        # Note: We must map the exclusion correctly. 
        # Since active_nodes[0] is 'curr', we exclude mins_in[0]
        sum_in = np.sum(mins_in[1:])
        
        # Sum of valid Outgoing edges (Everyone except Depot)
        # Since active_nodes[-1] is '0', we exclude mins_out[-1]
        sum_out = np.sum(mins_out[:-1])
        
        # Return average
        return float(0.5 * (sum_in + sum_out))

    # ==========================================
    # 2. Global Min Flow Bound (Max of Min-In/Out) [NEW]
    # ==========================================
    def h_global_min_flow(state):
        U = state[unvisited_var]
        curr = state[location_var]
        
        if not U and curr == 0:
            return 0.0

        # Bound A: Sum of minimum OUTGOING edges
        # We must leave 'curr' and every node in 'U'
        val_out = sum(min_outgoing_arr[u] for u in U)
        if curr != 0:
            val_out += min_outgoing_arr[curr]
            
        # Bound B: Sum of minimum INCOMING edges
        # We must enter '0' and every node in 'U'
        val_in = sum(min_incoming_arr[u] for u in U)
        if curr != 0: # If we aren't already at 0, we must eventually enter 0
            val_in += min_incoming_arr[0]
            
        # Return the tighter (maximum) of the two constraints
        return float(max(val_out, val_in))

    # ==========================================
    # 3. LP Relaxation Bound (On-the-fly)
    # ==========================================
    h_lp_relaxation = create_persistent_lp_relaxation_dual_bounds(metadata = metadata)

    # ==========================================
    # 4. MST Bound
    # ==========================================
    def h_mst(state):
        U = state[unvisited_var]
        if not U: return 0.0
        nodes = [0] + sorted(list(U))
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        mst = minimum_spanning_tree(sub_mat)
        return float(mst.sum())

    # ==========================================
    # 5. 1-Tree Bound
    # ==========================================
    def h_1tree(state):
        U = state[unvisited_var]
        if not U: return 0.0
        subset_nodes = sorted(list(U))
        
        depot_edges = sorted(cost_matrix[0, subset_nodes])
        e1 = depot_edges[0]
        e2 = depot_edges[1] if len(depot_edges) > 1 else 0.0
        
        if len(subset_nodes) > 1:
            sub_mat = cost_matrix[np.ix_(subset_nodes, subset_nodes)]
            mst_val = minimum_spanning_tree(sub_mat).sum()
        else:
            mst_val = 0.0 
        return float(mst_val + e1 + e2)

    # ==========================================
    # 6. Assignment Bound
    # ==========================================
    def h_assignment(state):
        U = state[unvisited_var]
        if not U: return 0.0
        nodes = [0] + sorted(list(U))
        sub_mat = cost_matrix[np.ix_(nodes, nodes)]
        assign_mat = sub_mat.astype(float).copy()
        # Fill diagonal with Infinity to forbid self-loops (i -> i)
        np.fill_diagonal(assign_mat, np.inf)
        # This finds the cheapest set of edges such that every row/col is used once
        row_ind, col_ind = linear_sum_assignment(assign_mat)
        return float(assign_mat[row_ind, col_ind].sum())

    # ==========================================
    # 7. Eigenvalue Bound
    # ==========================================
    def h_eigen(state):
        U = state[unvisited_var]
        nodes = [0] + sorted(list(U))
        N = len(nodes)
        if N < 2: return 0.0

        D_sub = cost_matrix[np.ix_(nodes, nodes)]
        one = np.ones((N, 1))
        P = np.eye(N) - (one @ one.T) / N
        M = -P @ D_sub @ P
        M = (M + M.T) / 2
        try:
            eigvals = np.flip(eigh(M)[0])
        except np.linalg.LinAlgError:
            return 0.0
        eigvals = eigvals[np.abs(eigvals) > 1e-9]
        coeffs = np.array([1 - np.cos(2 * np.pi * k / N) for k in range(1, N)])

        phi = 0.0
        if N > 1:
            if N % 2 == 1:
                num_terms = (N - 1) // 2
                if 2 * num_terms <= len(eigvals) and num_terms <= len(coeffs):
                     phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_terms + 1))
                else:
                    print(f"⚠️ Not enough eigenvalues/coefficients for odd N formula (subset {U})")
            else:
                num_sum_terms = N // 2 - 1
                if 2 * num_sum_terms < len(eigvals) and num_sum_terms <= len(coeffs):
                    phi = sum(coeffs[k-1] * (eigvals[2*k - 2] + eigvals[2*k - 1]) for k in range(1, num_sum_terms + 1))
                    if N-2 < len(eigvals):
                        phi += 2 * eigvals[N - 2]
                elif N > 1 and N-2 < len(eigvals):
                    phi = 2 * eigvals[N - 2]
                else:
                    print(f"⚠️ Not enough eigenvalues/coefficients for odd N formula (subset {U})")
        return float(phi)
    
    # Return valid registry
    dual_bound_dict = automatic_creation_of_dual_bounds_registry(locals())
    return dual_bound_dict
available_operations = available_operations

dual_bound_functions_registry = dual_bound_expression_function(creation_of_didp_model_function())

In [60]:
display(dual_bound_functions_registry)

{'h_degree_average': <function __main__.dual_bound_expression_function.<locals>.h_degree_average(state)>,
 'h_global_min_flow': <function __main__.dual_bound_expression_function.<locals>.h_global_min_flow(state)>,
 'h_lp_relaxation': <function __main__.create_persistent_lp_relaxation_dual_bounds.<locals>.h_lp_relaxation(state)>,
 'h_mst': <function __main__.dual_bound_expression_function.<locals>.h_mst(state)>,
 'h_1tree': <function __main__.dual_bound_expression_function.<locals>.h_1tree(state)>,
 'h_assignment': <function __main__.dual_bound_expression_function.<locals>.h_assignment(state)>,
 'h_eigen': <function __main__.dual_bound_expression_function.<locals>.h_eigen(state)>}

In [57]:
""" old h_lp_relaxation
def create_persistent_lp_relaxation_dual_bounds(metadata):
    # --- Extract Static Data ---
    n_nodes = metadata['num_nodes']
    unvisited_set_var = metadata['unvisited_var']
    location_var = metadata['location_var'] # Required to track current position in TSP
    dist_matrix = metadata['distance_matrix']
    

    # --- 1. SETUP MODEL (Runs Once) ---
    mdl = Model(name='TSP_Relaxation')
    
    # Optimization Parameters for Speed
    mdl.parameters.threads = 1
    mdl.parameters.lpmethod = 1 # Primal Simplex (Good for re-optimization)
    mdl.log_output = False      # Silence output

    # --- Create Variables ---
    # We pre-allocate variables for the full graph size (0..N)
    
    # x[i, j]: Flow variables (Continuous 0-1 for Relaxation)
    # Keys are tuples (i, j) - No vehicle index needed for TSP
    x = {}
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                x[(i, j)] = mdl.continuous_var(lb=0, ub=1, name=f'x_{i}_{j}')

    # u[i]: MTZ potential variables (representing order in tour)
    # Defined for all nodes, though typically used for non-depot nodes
    u = {i: mdl.continuous_var(lb=0, ub=n_nodes, name=f'u_{i}') for i in range(n_nodes)}

    # --- Static Objective ---
    # Minimize Sum of Costs
    obj_expr = mdl.sum(dist_matrix[i][j] * x[(i, j)] 
                    for i in range(n_nodes)
                    for j in range(n_nodes) if i != j)
    mdl.minimize(obj_expr)

    # --- 2. HEURISTIC FUNCTION (Runs per State) ---
    def h_lp_relaxation(state):

        # A. Identify Active Nodes (Sub-problem)
        # In DIDP, we must retrieve the value from the state using the variable object
        unvisited = state[unvisited_set_var]
        current_node = state[location_var]
        
        # Optimization: If solved (at depot with no unvisited), cost is 0
        if not unvisited and current_node == 0: 
            return 0.0

        # Active nodes = Current Location + Unvisited Set + Depot (0)
        # We treat the remaining path as a TSP tour on these specific nodes
        active_nodes_set = set(unvisited)
        active_nodes_set.add(current_node)
        active_nodes_set.add(0) # Ensure Depot is included
        
        active_nodes = list(active_nodes_set)
        n_active = len(active_nodes)
        
        # B. Update Variable Bounds (Active vs Inactive)
        # We "deactivate" edges involving inactive nodes by forcing UB=0
        active_set_lookup = set(active_nodes)
        
        for i in range(n_nodes):
            for j in range(n_nodes):
                if i == j: continue
                var = x[(i, j)]
                
                if i in active_set_lookup and j in active_set_lookup:
                    var.ub = 1 # Enable
                else:
                    var.ub = 0 # Disable
        
        # C. Re-Add Constraints for the Sub-problem
        mdl.clear_constraints()
        
        # (1) Assignment Constraints (Degree = 1)
        # For every active node, exactly one outgoing and one incoming edge
        # within the active subgraph.
        for i in active_nodes:
            # Outgoing = 1
            mdl.add_constraint(
                mdl.sum(x[(i, j)] for j in active_nodes if i != j) == 1
            )
            # Incoming = 1
            mdl.add_constraint(
                mdl.sum(x[(j, i)] for j in active_nodes if i != j) == 1
            )

        # (2) Miller-Tucker-Zemlin (MTZ) Subtour Elimination
        # Constraint: u[i] - u[j] + N * x[i,j] <= N - 1
        # Applied for all i, j in Active Set EXCEPT the Depot (node 0)
        for i in active_nodes:
            if i == 0: continue
            for j in active_nodes:
                if j == 0: continue
                if i != j:
                    mdl.add_constraint(
                        u[i] - u[j] + n_active * x[(i, j)] <= n_active - 1
                    )

        # D. Solve
        # We use a try-except block or check sol existence, as relaxation might be infeasible
        # if the graph is disconnected (though for TSP complete graph it's fine).
        sol = mdl.solve(url=None)
        
        if sol:
            return float(sol.objective_value)
        return 0.0

    return h_lp_relaxation
"""

' old h_lp_relaxation\ndef create_persistent_lp_relaxation_dual_bounds(metadata):\n    # --- Extract Static Data ---\n    n_nodes = metadata[\'num_nodes\']\n    unvisited_set_var = metadata[\'unvisited_var\']\n    location_var = metadata[\'location_var\'] # Required to track current position in TSP\n    dist_matrix = metadata[\'distance_matrix\']\n\n\n    # --- 1. SETUP MODEL (Runs Once) ---\n    mdl = Model(name=\'TSP_Relaxation\')\n\n    # Optimization Parameters for Speed\n    mdl.parameters.threads = 1\n    mdl.parameters.lpmethod = 1 # Primal Simplex (Good for re-optimization)\n    mdl.log_output = False      # Silence output\n\n    # --- Create Variables ---\n    # We pre-allocate variables for the full graph size (0..N)\n\n    # x[i, j]: Flow variables (Continuous 0-1 for Relaxation)\n    # Keys are tuples (i, j) - No vehicle index needed for TSP\n    x = {}\n    for i in range(n_nodes):\n        for j in range(n_nodes):\n            if i != j:\n                x[(i, j)] = mdl.c

# **2. Decoding operator**

In [62]:
def convert_chromosome_to_string_of_python_code(chromosome):
    """
    Translates a list-based chromosome (RPN) into a Python function definition string.
    """
    stack = []
    try:
        for gene in chromosome:
            if isinstance(gene, (int, float)):
                stack.append(str(gene))
            elif isinstance(gene, str):
                if gene == "ADD":
                    b, a = stack.pop(), stack.pop()
                    stack.append(f"({a} + {b})")
                elif gene == "SUBTRACT":
                    b, a = stack.pop(), stack.pop()
                    stack.append(f"({a} - {b})")
                elif gene == "MULTIPLY":
                    b, a = stack.pop(), stack.pop()
                    stack.append(f"({a} * {b})")
                elif gene == "MAX":
                    b, a = stack.pop(), stack.pop()
                    stack.append(f"max({a}, {b})")
                elif gene == "MIN":
                    b, a = stack.pop(), stack.pop()
                    stack.append(f"min({a}, {b})")
                elif gene == "PDIV":
                    b, a = stack.pop(), stack.pop()
                    # Protected Division Logic:
                    # If denominator (b) is close to 0, return 1.0 (identity), otherwise divide.
                    # We use an inline ternary operator for the string generation.
                    stack.append(f"({a} / {b} if abs({b}) > 1e-6 else 1.0)")
                else:
                    # It's a heuristic name like "h1"
                    stack.append(f"{gene}(state)")
        if len(stack) == 1:
            formula_body = stack.pop()
            # Define the standard name for our function
            func_name = "dual_bound_combination"
            code_string = f"def {func_name}(state):\n    return {formula_body}"
            return code_string, func_name
        else:
            return None, None
    except IndexError:
        return None, None

def validate_heuristic_presence(chromosome, dual_bound_functions_dict):
    """
    Checks if the chromosome contains at least one gene that is a valid heuristic name.
    Returns True if at least one heuristic (e.g., 'h1') is found.
    Returns False if the chromosome is purely constants/operators (e.g., [5, 2, 'ADD']).
    """
    # Get the set of valid heuristic names (keys from the registry)
    valid_heuristics = set(dual_bound_functions_dict.keys())
    
    # Check if any gene in the chromosome exists in that set
    for gene in chromosome:
        if gene in valid_heuristics:
            return True
            
    return False

def compile_chromosome_to_useable_function(chromosome_fitness_dict, dual_bound_functions_dict, print_code):
    """
    Converts a chromosome list into a real, callable Python function.
    Args:
        chromosome: The list (RPN)
        dual_bound_functions_dict: Dict of actual functions available to the code
        (e.g., {'h1': h1, 'h2': h2})
    """
    
    chromosome = extract_chromosome_from_chromosome_fitness_dict(chromosome_fitness_dict)
    
    if not validate_heuristic_presence(chromosome, dual_bound_functions_dict):
        # This will trigger the 'except' block in your EA loop
        raise ValueError("Invalid Chromosome: No heuristics found (Pure Constant).")
    
    # 1. Generate the code string
    code_string, func_name = convert_chromosome_to_string_of_python_code(chromosome)
    if code_string is None:
        raise ValueError("Invalid Chromosome")
    if print_code:
      print(f"Generated Code:\n{code_string}\n")

    # 2. Prepare the execution namespace
    # We create a dictionary that holds all the functions our new code needs.
    # This acts like the 'globals()' for the new function.
    execution_namespace = dual_bound_functions_dict.copy()

    # 3. Compile and Execute the string
    # This runs the 'def dual_bound_combination...' string, creating the function
    # inside the execution_namespace dict.
    exec(code_string, execution_namespace)

    # 4. Retrieve the live function object
    callable_function = execution_namespace[func_name]

    return callable_function

# **3. Encoding operator**

## General generator

In [66]:
def generate_valid_chromosome(dual_bound_functions_dict, 
                              LB_range_of_constant = LB_range_of_constant,
                              UB_range_of_constant = UB_range_of_constant, 
                              available_operations = available_operations):
    """
    Generates a valid random chromosome using keys from the registry.
    Now supports multiplying bounds together!
    """
    # Extract the keys (for instance ["h1", "h2", "h3"]) dynamically from the dictionary
    available_dual_bound_functions = list(dual_bound_functions_dict.keys())

    # 1. Create initial blocks (Terminals) by iteration through each heuristic to add in coefficients
    blocks = []
    for h_name in available_dual_bound_functions:
        if random.random() < 0.5:
            # Weighted block: [coef, h, "MULTIPLY"] -> resulting in (coef * h)
            coef = round(random.uniform(LB_range_of_constant, UB_range_of_constant), 2)
            blocks.append([coef, h_name, "MULTIPLY"])
        else:
            # Raw block: [h]
            blocks.append([h_name])

    # 2. Merge blocks until one remains
    #Create a copy for the blocks of each heuristic function
    current_pool = blocks.copy()

    while len(current_pool) > 1:
        # Pick two random blocks from the pool
        # Ensure there are at least 2 blocks to sample from
        if len(current_pool) < 2:
            break

        idx1, idx2 = random.sample(range(len(current_pool)), 2)

        # Pop in descending order to handle index shifts
        right = current_pool.pop(max(idx1, idx2))
        left = current_pool.pop(min(idx1, idx2))

        # Merge with random operator
        op = random.choice(available_operations)

        # Structure: [Left Block] + [Right Block] + [Operator]
        merged = left + right + [op]

        current_pool.append(merged)

    return current_pool[0]

## Ramped half-and-half generator

In [67]:
def generate_ramped_half_and_half(dual_bound_functions_dict, 
                                  LB_range_of_constant = LB_range_of_constant ,
                                  UB_range_of_constant = UB_range_of_constant,
                                  min_chromosome_length = min_chromosome_length, 
                                  max_chromosome_length = max_chromosome_length, 
                                  available_operations = available_operations):
    """
    Generates a chromosome (RPN list) using the Ramped Half-and-Half method.

    Args:
        min_depth = min_chromosome_length (int): Minimum tree depth.
        max_depth = max_chromosome_length (int): Maximum tree depth.
    """

    # 1. RAMPED: Pick a random max depth for this individual
    # This ensures the population has a mix of shallow and deep trees.
    target_depth = random.randint(min_chromosome_length, max_chromosome_length)

    # 2. HALF-AND-HALF: Choose method
    # 50% chance for "GROW", 50% chance for "FULL"
    method = "GROW" if random.random() < 0.5 else "FULL"

    # 3. Generate
    # We start at depth 0
    chromosome = generate_rpn_tree_recursive(
        current_depth = 0,
        max_node_depth = target_depth,
        method = method,
        dual_bound_functions_dict = dual_bound_functions_dict,
        LB_range_of_constant = LB_range_of_constant,
        UB_range_of_constant = UB_range_of_constant,
        available_operations = available_operations
    )
    return chromosome

## Integrated generator

In [68]:
def generate_combined_chromosome(dual_bound_functions_dict, 
                                LB_range_of_constant, UB_range_of_constant,
                                min_chromosome_length, max_chromosome_length, 
                                available_operations):
    """
    Generates a chromosome using a combined strategy:
    - 70% chance to use the General Generator (generate_valid_chromosome)
    - 30% chance to use the Ramped Half-and-Half Generator (generate_ramped_half_and_half)
    """
    if random.random() < 0.5:
        # Use General Generator
        chromosome = generate_valid_chromosome(dual_bound_functions_dict = dual_bound_functions_dict, 
                                            LB_range_of_constant = LB_range_of_constant, 
                                            UB_range_of_constant = UB_range_of_constant, 
                                            available_operations = available_operations
                                            )
        chromosome_fitness_dict = {'chromosome': chromosome, 'fitness': 0}
        return chromosome_fitness_dict
    else:
        # Use Ramped Half-and-Half Generator
        chromosome = generate_ramped_half_and_half(dual_bound_functions_dict = dual_bound_functions_dict, 
                                                LB_range_of_constant = LB_range_of_constant,
                                                UB_range_of_constant = UB_range_of_constant, 
                                                min_chromosome_length = min_chromosome_length, 
                                                max_chromosome_length = max_chromosome_length, 
                                                available_operations = available_operations
                                                )
        chromosome_fitness_dict = {'chromosome': chromosome, 'fitness': 0}
        return chromosome_fitness_dict

# **4. Fitness calculation operator**

In [70]:
def print_comprehensive_solver_report(cost, is_optimal, generated, expanded, timing_info):
    """
    Prints a standardized, high-detail report of the solver run.
    
    Args:
        cost (float): The solution cost found.
        is_optimal (bool): Whether the solution is proven optimal.
        generated (int): Number of nodes generated.
        expanded (int): Number of nodes expanded.
        timing_info (dict): Dictionary containing detailed timing breakdowns.
    """
    # 1. Header
    print("\n" + "="*60)
    print(f"{'📋 SOLVER RUN REPORT':^60}")
    print("="*60)

    # 2. Solution Quality & Search Stats
    opt_status = "✅ Proven Optimal" if is_optimal else "⚠️  Suboptimal / Timeout"
    print(f"🎯 SOLUTION STATUS:")
    print(f"   • Cost:              {cost}")
    print(f"   • Status:            {opt_status}")
    print(f"   • Nodes Generated:   {generated:,}")
    print(f"   • Nodes Expanded:    {expanded:,}")
    
    if expanded > 0:
        ratio = generated / expanded
        print(f"   • Branching Factor:  ~{ratio:.2f}")
    print("-" * 60)

    # 3. Performance Analysis (if timing info is present)
    if timing_info:
        # Normalize keys (handle variations if any)
        t_total = timing_info.get("total_wall_clock", timing_info.get("total_duration", 0))
        t_cabs = timing_info.get("pure_cabs_time", 0)
        t_bridge = timing_info.get("total_bridge_time", timing_info.get("bridge_time", 0))
        t_calc = timing_info.get("python_dual_bound_calc_time", timing_info.get("python_calc", 0))
        t_switch = timing_info.get("switching_overhead", 0)
        n_calls = timing_info.get("total_calls", timing_info.get("calls", 0))

        # Calculate Percentages
        pct_cabs = (t_cabs / t_total * 100) if t_total > 0 else 0
        pct_bridge = (t_bridge / t_total * 100) if t_total > 0 else 0
        pct_calc = (t_calc / t_bridge * 100) if t_bridge > 0 else 0
        pct_switch = (t_switch / t_bridge * 100) if t_bridge > 0 else 0

        print(f"⏱️  TIME DISTRIBUTION (Total: {t_total:.4f}s):")
        
        # Rust vs Bridge Bar
        print(f"   [Pure CABS time: {pct_cabs:5.1f}%] 🆚 [Bridge time: {pct_bridge:5.1f}%]")
        
        print(f"\n   1. 🟢 Pure CABS (Search):    {t_cabs:.4f} s")
        if expanded > 0:
            print(f"      └─ Average time per expanded state:   {(t_cabs/expanded*1000):.4f} ms")

        print(f"   2. 🔴 Total Bridge Time:     {t_bridge:.4f} s")
        print(f"      ├─ 🐍 Dual Bound Calculation Time: :     {t_calc:.4f} s  ({pct_calc:5.1f}% of bridge time)")
        print(f"      └─ 🌉 Switching Time:  {t_switch:.4f} s  ({pct_switch:5.1f}% of bridge time)")
        
        print("-" * 60)
        
        # Per-Call Stats
        if n_calls > 0:
            print(f"📊 PYTHON DUAL BOUND CALL STATS:")
            print(f"   • Total Calls:       {n_calls:,}")
            print(f"   • Avg Bridge Time:   {(t_bridge/n_calls*1000):.4f} ms")
            print(f"   • Avg Pure Calc:     {(t_calc/n_calls*1000):.4f} ms")
            print(f"   • Avg Switching:     {(t_switch/n_calls*1000):.4f} ms")
            
            # Heuristic Rate
            rate = n_calls / t_total if t_total > 0 else 0
            print(f"   • Throughput:        {rate:,.0f} calls/sec")

    print("="*60 + "\n")

def combining_modified_didppy_solver_with_chromosome(chromosome, 
                                                    didp_model_registry, 
                                                    dual_bound_expression_function, 
                                                    solver_time_limit = None,
                                                    output_other_result = False,
                                                    print_timing_stats = False):
    """
    Runs the DIDP solver with a flexible, evolved heuristic.
    
    Args:
        chromosome (list): The RPN list (e.g., ['h1', 'h2', 'ADD']).
        didp_model_registry (func): Returns (model, model_vars).
        dual_bound_expression_function (func): Accepts (model, model_vars) and returns 
                                the dual_bound_registry dict.
        solver_time_limit (float): Time limit for the solver.
        output_other_result (bool): If True, returns extended stats (cost, is_optimal, gen, exp, timings).
        print_timing_stats (bool): If True, enables Rust-side timing and parses the output file.
    """
    
    # --- 1. Create Fresh Model Instance ---
    didp_bundle = didp_model_registry() 
    didp_model = didp_bundle[0]
    
    # --- 2. Create Linked Heuristics ---
    dual_bound_registry = dual_bound_expression_function(didp_bundle)
    
    # --- 3. Compile Combined Bound ---
    try:
        temp_individual_dict = {'chromosome': chromosome}
        combined_bound_func = compile_chromosome_to_useable_function(
            chromosome_fitness_dict=temp_individual_dict, 
            dual_bound_functions_dict= dual_bound_registry, 
            print_code=False
        )
    except Exception as e:
        print(f"Heuristic Compilation Failed: {e}")
        return float('inf')

    # 🟢 SETUP PYTHON TIMING
    # 🟢 OPTIMIZATION: Conditionally define the wrapper
    # We define distinct functions so the "False" case has ZERO timing overhead.
    
    py_stats = None # Default to None

    if print_timing_stats:
        # --- A. SLOW PATH (With Timing) ---
        py_stats = {"dual_bound_calc_time": 0.0, "calls": 0}
        
        def safe_dual_bound(state):
            t0 = time.perf_counter()
            try:
                val = combined_bound_func(state)
                res = float(val)
            except Exception:
                res = 0.0
            t1 = time.perf_counter()
            
            # Accumulate stats
            py_stats["dual_bound_calc_time"] += (t1 - t0)
            py_stats["calls"] += 1
            return res
            
    else:
        # --- B. FAST PATH (Production Mode) ---
        # No timers, no dictionary lookups. Just execution.
        def safe_dual_bound(state):
            try:
                # Direct return is slightly faster
                return float(combined_bound_func(state))
            except Exception:
                return 0.0
    

    # --- 5. Run Solver ---
    try:  
        solver = m_dp.CustomDualBoundCABSv1(
            didp_model, 
            dual_bound_func=safe_dual_bound, 
            quiet=True,
            time_limit = solver_time_limit,
            print_timing_stats = print_timing_stats
        )
        
        # Measure Total Wall Clock
        start_time = time.time()
        solution = solver.search()
        end_time = time.time()
        
        total_duration = end_time - start_time
        timing_info = None

        # 🟢 PARSE & CALCULATE BREAKDOWN
        if print_timing_stats:
            log_filename = "Python_and_Rust_bridge_time_stats.txt"
            if os.path.exists(log_filename):
                try:
                    with open(log_filename, 'r') as f:
                        content = f.read()
                    
                    #🟢 FIX: Regex now matches "s" instead of "seconds"
                    rust_match = re.search(r"Total time in Python:\s*([\d\.]+)\s*s", content)
                    calls_match = re.search(r"Total calls:\s*(\d+)", content)
                    
                    if rust_match:
                        # 1. Get raw values
                        total_bridge_time = float(rust_match.group(1)) # T_total_bridge
                        python_dual_bound_calc_time = py_stats["dual_bound_calc_time"]       # T_calc
                        
                        # 2. Calculate Components
                        # Switching Cost = (Total time Rust spent waiting) - (Time Python spent calculating)
                        switching_overhead = max(0.0, total_bridge_time - python_dual_bound_calc_time)
                        
                        # Pure CABS = (Total Wall Clock) - (Total Bridge Time)
                        pure_cabs_time = max(0.0, total_duration - total_bridge_time)
                        
                        total_calls = int(calls_match.group(1)) if calls_match else 0
                        
                        timing_info = {
                            "total_duration": total_duration,
                            "total_bridge_time": total_bridge_time,
                            "pure_cabs_time": pure_cabs_time,
                            "switching_overhead": switching_overhead,
                            "python_dual_bound_calc_time": python_dual_bound_calc_time,
                            "total_calls": total_calls
                        }
                except Exception as e:
                    print(f"Failed to parse timing stats: {e}")

        # --- 6. Return Results ---
        if solution.cost is not None:
            if output_other_result and (print_timing_stats) :
                didp_model_dual_bound_cost = solution.cost
                solution_optimality = solution.is_optimal
                number_generated = solution.generated
                number_expanded = solution.expanded
                timming_info = timing_info
                return print_comprehensive_solver_report(
                    cost=didp_model_dual_bound_cost, 
                    is_optimal=solution_optimality, 
                    generated=number_generated, 
                    expanded=number_expanded, 
                    timing_info=timing_info)
            elif output_other_result and (not print_timing_stats) :
                return (solution.cost, solution.is_optimal, solution.generated, solution.expanded)
            else:
                return solution.cost
        else:
            if output_other_result:
                return float('inf'), False, 0, 0, timing_info
            else:
                return float('inf')

    except Exception as e:
        print(f"Solver Error: {e}")
        if output_other_result:
            return float('inf'), False, 0, 0, None
        return float('inf')

def chromosome_fitness_dict_evaluation(chromosome_fitness_dict, 
                                    didp_model_registry, 
                                    dual_bound_expression_function, 
                                    reference_point = OPTIMAL_COST_REFERENCE,
                                    solver_time_limit = None,
                                    output_other_result = False,
                                    print_timing_stats = False):
    """
    Evaluates a chromosome by running the DIDP solver and comparing the result to a reference point.
    Updates the 'fitness' key in the input dictionary.

    Args:
        chromosome_fitness_dict (dict): {'chromosome': [...], 'fitness': None}
        didp_model_registry (func): Function to create DIDP model & meatdata (variables, state variables).
        dual_bound_functions_expression (func): Function to create dual bound registry.
        reference_point (float): The known optimal cost (Ground Truth).
        
    Returns:
        dict: The updated chromosome_fitness_dict with calculated fitness.
    """
    
    # 1. Extract the chromosome list
    chromosome = chromosome_fitness_dict.get('chromosome')
    if not chromosome:
        chromosome_fitness_dict['fitness'] = float('inf')
        return chromosome_fitness_dict

    # 2. Run the Solver Pipeline
    # We use the combining function to get the actual objective value found by the solver
    solver_result_cost = combining_modified_didppy_solver_with_chromosome(
        chromosome = chromosome,
        didp_model_registry = didp_model_registry,
        dual_bound_expression_function = dual_bound_expression_function,
        solver_time_limit = solver_time_limit
    )

    # 3. Calculate Deviation and Fitness
    # Handle Solver Failure (Infeasible or Error)
    if solver_result_cost == float('inf'):
        fitness = float('inf')
    
    else:
        # Calculate deviation from the reference (optimal) point
        if reference_point != 0:
            deviation = abs(solver_result_cost - reference_point) / abs(reference_point)
        else:
            deviation = abs(solver_result_cost - reference_point)

        # 4. Apply Penalties based on Constraints
        # CONSTRAINT: The solver cost should match the reference (Optimal).
        # If Solver Cost > Reference: Heuristic was likely inadmissible (pruned the optimal path).
        # If Solver Cost == Reference: Heuristic was safe. Fitness is based on deviation (0) or efficiency.
        
        tolerance = 1e-6 # For float comparison
        
        if solver_result_cost <= reference_point + tolerance:
            # Case 1: Success (Optimal Solution Found)
            # Fitness is the deviation (ideally 0). 
            # You could add logic here to reward fewer expanded nodes if available.
            fitness = deviation
        else:
            # Case 2: Failure (Suboptimal Solution Found)
            # The heuristic likely overestimated and pruned the optimal path.
            # Apply massive penalty.
            fitness = deviation

    # 5. Update and Return
    chromosome_fitness_dict['fitness'] = fitness
    
    return chromosome_fitness_dict

# **5. Population initilization**

In [95]:
def initialize_list_of_chromosome_fitness_dictionary(list_size, 
                                                    dual_bound_functions_dict, 
                                                    didp_model_registry, 
                                                    dual_bound_expression_function,
                                                    LB_range_of_constant = LB_range_of_constant, 
                                                    UB_range_of_constant = UB_range_of_constant,
                                                    min_chromosome_length = min_chromosome_length,
                                                    max_chromosome_length = max_chromosome_length,
                                                    available_operations = available_operations,
                                                    reference_point = OPTIMAL_COST_REFERENCE,
                                                    solver_time_limit = None):
    """
    Creates a population list where each element is a dictionary holding the chromosome 
    AND its evaluated fitness.
    
    Args:
        list_size (int): Number of individuals to generate.
        didp_model_registry (func): Factory for the DIDP model.
        dual_bound_expression_function (func): Factory for the heuristics.
        reference_point (float): Optimal cost for fitness calculation.
        [Other args match the generator inputs]

    Returns:
        list: A list of fully evaluated dicts: [{'chromosome': [...], 'fitness': 0.5}, ...]
    """
    population = []
    
    for _ in range(list_size):
        # 1. Generate Chromosome
        newly_generated_chromosome_fitness_dict = generate_combined_chromosome(
            dual_bound_functions_dict = dual_bound_functions_dict, 
            LB_range_of_constant = LB_range_of_constant, 
            UB_range_of_constant = UB_range_of_constant,
            min_chromosome_length = min_chromosome_length, 
            max_chromosome_length = max_chromosome_length, 
            available_operations = available_operations
        )

        
        # 3. Evaluate Fitness Immediately
        evaluated_newly_generated_chromosome_fitness_dict = chromosome_fitness_dict_evaluation(
            chromosome_fitness_dict =newly_generated_chromosome_fitness_dict, 
            didp_model_registry = didp_model_registry, 
            dual_bound_expression_function = dual_bound_expression_function, 
            reference_point = reference_point,
            solver_time_limit = solver_time_limit
        )
        
        population.append(evaluated_newly_generated_chromosome_fitness_dict)

    return population

# **6. Parent selection operator**

In [97]:
def parents_selection(population, 
                    tournament_size=tournament_size, 
                    tournament_probability=tournament_probability):
    """
    Selects a single parent using a Tournament Selection.
    
    Features:
    - Random Tournament Size: Defaults to random(2, 10) if not specified.
    - While Loop Logic: Uses manual loops for selection and knockout as requested.
    """

        
    # Safety: Cannot have a tournament larger than the population
    actual_size = min(len(population), tournament_size)
    
    # 1. Select Candidates (Using While Loop)
    list_of_candidates = []
    
    # 1. Select Candidates Efficiently
    # random.sample picks 'actual_size' unique elements without replacement
    list_of_candidates = random.sample(population, actual_size)
            
    # 3. Apply Tournament Logic
    if random.random() < tournament_probability:
        winner_of_tournament = list_of_candidates.copy()
        
        # Knockout Loop: Fight until 1 remains
        while len(winner_of_tournament) > 1:
            # Pick two random fighters
            idx_1 = random.randint(0, len(winner_of_tournament) - 1)
            candidate_1 = winner_of_tournament[idx_1]
            winner_of_tournament.remove(candidate_1)
            
            idx_2 = random.randint(0, len(winner_of_tournament) - 1)
            candidate_2 = winner_of_tournament[idx_2]
            winner_of_tournament.remove(candidate_2)
            
            # Extract Fitness (Handle None as Infinity for minimization)
            f1 = candidate_1['fitness'] if candidate_1['fitness'] is not None else float('inf')
            f2 = candidate_2['fitness'] if candidate_2['fitness'] is not None else float('inf')
            
            # Compare (Lower Fitness is Better)
            if f1 < f2:
                winner_of_tournament.append(candidate_1)
            else:
                winner_of_tournament.append(candidate_2)
        
        # The champion
        parent = winner_of_tournament[0]
        
    else:
        # --- Random Mode (20% chance) ---
        parent = random.choice(list_of_candidates)
        
    return parent

# **7. Crossover operator**

## One-point crossover

### Utility functions of 1-point crossover

In [103]:
def construct_the_lookup_map_for_rpn(chromosome_rpn_topology):
    """
    Creates a map {end_index: (start, end, type, depth)}
    """
    lookup = {}
    for idx, node_data in chromosome_rpn_topology.items():
        start, end = node_data['subtree_range']
        node_type = node_data['type']
        depth = node_data['depth'] # <--- Retrieve it
        
        # Store as 4-element tuple
        lookup[end] = (start, end, node_type, depth)
        
    return lookup

def map_chromosome_fitness_dict_to_rpn_topology(chromosome_fitness_dict, 
                                                available_operations = available_operations):
    """
    Traverses RPN *Forwards* to map indices to tree structure.
    
    Why Forward?
    RPN is 'Left Child' -> 'Right Child' -> 'Operator'.
    By scanning forward, we ensure children are already on the stack 
    when we hit an operator.
    
    Returns a dict: index -> {'type', 'subtree_range', 'left_child', 'right_child'}
    """
    topology = {}
    stack = []
    chromosome = extract_chromosome_from_chromosome_fitness_dict(chromosome_fitness_dict)
    for i, gene in enumerate(chromosome):
        arity = get_rpn_node_arity(gene, available_operations)
        node_value = gene 
        
        if arity == 0:
            subtree_start_location = i
            stack.append((i, subtree_start_location))
            
            topology[i] = {
                'type': 'TERMINAL', 
                'value': node_value, 
                'subtree_range': (i, i),
                'depth': len(stack)  # Depth is current stack size
            }
            
        elif arity == 2:
            if len(stack) < 2: return {} 
            
            right_idx, right_start = stack.pop()
            left_idx, left_start = stack.pop()
            subtree_range = (left_start, i)
            current_depth = len(stack) + 1
            
            topology[i] = {
                'type': 'FUNCTION', 
                'value': node_value,
                'subtree_range': subtree_range,
                'left_child': left_idx,
                'right_child': right_idx,
                'depth': current_depth
            }
            stack.append((i, left_start))
            
    return topology

def find_homologous_pairs(parent1_dict, parent2_dict, 
                        available_operations):
    """
    Identifies indices in Parent 1 and Parent 2 that share the same topology 
    by traversing both trees from the Root down.
    """
    # 1. Extract raw lists
    p1_chromosome = extract_chromosome_from_chromosome_fitness_dict(chromosome_fitness_dict = parent1_dict)
    p2_chromosome = extract_chromosome_from_chromosome_fitness_dict(chromosome_fitness_dict = parent2_dict)
    
    # 2. Build Topology Maps (Index -> Node Data)
    # FIX: Call the correct mapping function first!
    p1_topology = map_chromosome_fitness_dict_to_rpn_topology(chromosome_fitness_dict =parent1_dict,
                                                            available_operations = available_operations)
    p2_topology = map_chromosome_fitness_dict_to_rpn_topology(chromosome_fitness_dict = parent2_dict, 
                                                            available_operations = available_operations)
    
    matching_pairs = []
    
    # 3. Initialize Traversal Queue (Start at Roots)
    root_p1 = len(p1_chromosome) - 1
    root_p2 = len(p2_chromosome) - 1
    
    queue = [(root_p1, root_p2)]
    
    # 4. BFS Traversal
    while queue:
        idx1, idx2 = queue.pop(0)
        
        # Retrieve node structure data
        node1 = p1_topology.get(idx1)
        node2 = p2_topology.get(idx2)
        
        if not node1 or not node2: 
            continue
        
        # Check Homology
        if node1['type'] == node2['type']:
            matching_pairs.append((idx1, idx2))
            
            if node1['type'] == 'FUNCTION':
                queue.append((node1['left_child'], node2['left_child']))
                queue.append((node1['right_child'], node2['right_child']))
                
    return matching_pairs, p1_topology, p2_topology

### Core function of 1-point crossover

In [104]:
def one_point_crossover_two_offspring(parent1_dict, parent2_dict, 
                                    dual_bound_functions_registry, 
                                    homology_1_point_crossover_probability = homology_1_point_crossover_probability,
                                    available_operations = available_operations):
    """
    Performs 1-Point Crossover.
    Priority: Try to swap at a Homologous Point (Matching Structure).
    Fallback: If no homology, swap at random Function points in both parents.
    Swaps the PREFIX (Head) of the chromosomes.
    """
    p1_chromosome = extract_chromosome_from_chromosome_fitness_dict(parent1_dict)
    p2_chromosome = extract_chromosome_from_chromosome_fitness_dict(parent2_dict)
    
    # 1. Find Homologous Pairs (Common Regions) using the Helper
    matching_pairs, p1_topology, p2_topology= find_homologous_pairs(parent1_dict = parent1_dict,
                                                                    parent2_dict = parent2_dict ,
                                                                    available_operations = available_operations)

    # 2. Select Crossover Points
    if matching_pairs and random.random() < homology_1_point_crossover_probability:
        # CASE A: Homology Found - Pick matching structure
        cp1, cp2 = random.choice(matching_pairs)
    else:
        # CASE B: Fallback - Random Functions with MATCHING DEPTH
        p1_lookup = construct_the_lookup_map_for_rpn(p1_topology)
        p2_lookup = construct_the_lookup_map_for_rpn(p2_topology)
        
        # Filter: Must be FUNCTION
        # lookup items: (start, end, type, depth)
        p1_funcs = [idx for idx, data in p1_lookup.items() if data[2] == 'FUNCTION']
        p2_funcs = [idx for idx, data in p2_lookup.items() if data[2] == 'FUNCTION']
        
        # Find compatible pairs (Same Stack Depth)
        compatible_pairs = []
        for i1 in p1_funcs:
            depth1 = p1_lookup[i1][3] # Index 3 is Depth
            for i2 in p2_funcs:
                depth2 = p2_lookup[i2][3]
                if depth1 == depth2:
                    compatible_pairs.append((i1, i2))
    
        if compatible_pairs:
            cp1, cp2 = random.choice(compatible_pairs)
        else:
            # Absolute fallback: Roots always have depth 1
            cp1 = len(p1_chromosome) - 1
            cp2 = len(p2_chromosome) - 1
    
    # 3. Perform Prefix Swap (Head Swap)
    # Offspring 1: Head of P2 + Tail of P1
    offspring1 = p2_chromosome[:cp2+1] + p1_chromosome[cp1+1:]
    
    # Offspring 2: Head of P1 + Tail of P2
    offspring2 = p1_chromosome[:cp1+1] + p2_chromosome[cp2+1:]
    
    # 4. Validate & Compile
    offspring_list = []
    fallback_candidate_list  = [p2_chromosome, p1_chromosome]
    for i, chromosome in enumerate([offspring1, offspring2]):
        chromosome_fitness_dict = {'chromosome': chromosome, 'fitness': 0}
        try:
            func = compile_chromosome_to_useable_function(chromosome_fitness_dict = chromosome_fitness_dict, 
                                                        dual_bound_functions_dict = dual_bound_functions_registry, 
                                                        print_code=False)
            offspring_list.append(chromosome)
        except Exception as e:
            print(f"Offspring {i+1} of one-point crossover is invalid: {e}")
            # Optionally, could append None or a default value
            fallback_candidate = random.choice(fallback_candidate_list)
            fallback_candidate_list.remove(fallback_candidate)
            print(f"Using fallback parent chromosome for Offspring {i+1} of this one-point crossover instance.")
            offspring_list.append(fallback_candidate)
    return offspring_list

## Subtree crossover

In [105]:
def subtree_crossover(parent1_dict, parent2_dict, 
                    dual_bound_functions_registry, 
                    subtree_crossover_probability=subtree_crossover_probability, 
                    available_operations=available_operations):
    """
    Performs subtree crossover between two parents.
    Input: Dicts {'chromosome': [...], 'fitness': ...}
    Output: A new child chromosome (list).
    """
    p1_chromosome = extract_chromosome_from_chromosome_fitness_dict(parent1_dict)
    p2_chromosome = extract_chromosome_from_chromosome_fitness_dict(parent2_dict)
    
    # --- Step 1: Identify Structures ---
    p1_structure = analyzing_chromosome_based_on_rpn_structure(chromosome_fitness_dict = parent1_dict,
                                                            available_operations = available_operations)
    p2_structure = analyzing_chromosome_based_on_rpn_structure(chromosome_fitness_dict= parent2_dict,
                                                            available_operations = available_operations)
    
    # --- Step 2: Choose Point for Parent 1 (The Receiver) ---
    # We decide whether we want to replace a Function or a Terminal in P1
    if random.random() < subtree_crossover_probability and p1_structure["FUNCTIONS"]:
        # 90% chance: Replace a Function tree
        candidate1 = p1_structure["FUNCTIONS"]
    else:
        # 10% chance: Replace a Terminal leaf
        candidate1 = p1_structure["TERMINALS"]    
    # Fallback: if P1 has no functions (it's just a leaf), must pick terminal
    # Filter out the root (which is always the last index in RPN)
    root_index = len(p1_chromosome) - 1
    valid_candidates = [
        (s, e) for (s, e) in candidate1 
        if e != root_index  # Don't let the end of the subtree be the root
    ]

    if valid_candidates:
        p1_start, p1_end = random.choice(valid_candidates)
    else:
        # Fallback: If no other options exist (e.g., P1 is already just a leaf), 
        # you might have to pick the root or skip crossover.
        candidate1 = p1_structure["TERMINALS"]   
        p1_start, p1_end = random.choice(candidate1)
    
    # --- Step 3: Choose Point for Parent 2 (The Donor) ---
    # The donor part can be anything (Function or Terminal), 
    # as long as it produces 1 value.
    if random.random() < subtree_crossover_probability and p2_structure["FUNCTIONS"]:
        # 90% chance: Replace a Function tree
        candidate2 = p2_structure["FUNCTIONS"]
    else:
        # 10% chance: Replace a Terminal leaf
        candidate2 = p2_structure["TERMINALS"]  
    if not candidate2: candidate2 = p2_structure["TERMINALS"]   
    p2_start, p2_end = random.choice(candidate2)
    
    # --- Step 4: Process Crossover ---
    # Construct Offspring: P1_before_crossover_point + P2_Subtree + P1_after_crossover_point
    
    # 1. Part of P1 before the cut
    head_of_parent1_chromosome = p1_chromosome[:p1_start]
    
    # 2. The subtree from P2
    donor_gene = p2_chromosome[p2_start : p2_end+1]
    
    # 3. Part of P1 after the cut
    tail_of_parent1_chromosome = p1_chromosome[p1_end+1:]
    
    offspring_chromosome = head_of_parent1_chromosome + donor_gene + tail_of_parent1_chromosome
    
    # --- Step 5: Safety validation ---
    try:
        offspring_chromosome_function = compile_chromosome_to_useable_function(
            {'chromosome': offspring_chromosome, 'fitness':0}, 
            dual_bound_functions_dict = dual_bound_functions_registry, 
            print_code=False
        )
    except Exception as e:
        print(f"Subtree crossover produced invalid offspring: {e}")
        print(f"Using fallback parent chromosome for this subtree crossover.")
        # In case of invalid offspring, return parent1's chromosome as fallback
        return random.choice([parent1_dict['chromosome'], parent2_dict['chromosome']])

    return offspring_chromosome

## Uniform crossover

In [106]:
def uniform_crossover_weighted_protected(parent1_dict, parent2_dict, 
                                        dual_bound_functions_registry, 
                                        uniform_crossover_probability=uniform_crossover_probability):
    """
    Performs Uniform Crossover with Atomic Block Swapping.
    
    - Standard: Swaps compatible genes (Term<->Term, Func<->Func).
    - Protected Blocks: If BOTH parents have a [Num, Str, MUL] block at the same spot,
    swaps the ENTIRE block as a single unit (Safer).
    - Unmatched Blocks: Protects the 'MULTIPLY' operator but allows constituents to swap.
    """
    p1_chromosome = extract_chromosome_from_chromosome_fitness_dict(parent1_dict)
    p2_chromosome = extract_chromosome_from_chromosome_fitness_dict(parent2_dict)
    
    # 1. Analyze Structure
    p1_structure = analyzing_chromosome_based_on_rpn_structure(chromosome_fitness_dict = parent1_dict,
                                                            available_operations = available_operations)
    p2_structure = analyzing_chromosome_based_on_rpn_structure(chromosome_fitness_dict= parent2_dict,
                                                            available_operations = available_operations)
    
    # 2. Build Sets of "Protected Operators" (The MULTIPLY index)
    protected_indices_p1 = set()
    for start, end in p1_structure["TERMINALS"]:
        if (end - start) == 2: 
            protected_indices_p1.add(end)
            
    protected_indices_p2 = set()
    for start, end in p2_structure["TERMINALS"]:
        if (end - start) == 2:
            protected_indices_p2.add(end)
            
    # 3. Iteration Setup
    min_len = min(len(p1_chromosome), len(p2_chromosome))
    offspring1 = p1_chromosome.copy()
    offspring2 = p2_chromosome.copy()
    
    i = 0
    # Stop before the Root (last index) to protect it
    while i < min_len - 1:
        
        # --- LOGIC 1: ATOMIC BLOCK SWAP (The "Safer" Logic) ---
        # Check if we are at the start of a Protected Block in BOTH parents.
        # A block [Num, Str, MUL] starting at 'i' ends at 'i+2'.
        block_end_index = i + 2
        
        # Ensure we don't go out of bounds
        if block_end_index < (min_len - 1):
            is_block_p1 = (block_end_index in protected_indices_p1)
            is_block_p2 = (block_end_index in protected_indices_p2)
            
            if is_block_p1 and is_block_p2:
                # Both have a full TERMINAL block. Swap the WHOLE BLOCK as one unit.
                if random.random() < uniform_crossover_probability:
                    # Swap Weight (i)
                    offspring1[i], offspring2[i] = offspring2[i], offspring1[i]
                    # Swap Heuristic (i+1)
                    offspring1[i+1], offspring2[i+1] = offspring2[i+1], offspring1[i+1]
                    # Swap Operator (i+2) - They are identical MULs, but good for consistency
                    offspring1[i+2], offspring2[i+2] = offspring2[i+2], offspring1[i+2]
                
                # Advance 3 steps (Skip the constituents we just handled)
                i += 3
                continue
        # --- LOGIC 2: STANDARD CONSTITUENT SWAP ---
        gene1 = p1_chromosome[i]
        gene2 = p2_chromosome[i]
        
        arity1 = get_rpn_node_arity(gene = gene1, available_operations = available_operations)
        arity2 = get_rpn_node_arity(gene = gene2, available_operations = available_operations)
        
        # Only swap if Arity matches
        if arity1 == arity2:
            # Protection Check: Don't swap if one is a Protected Operator and the other isn't
            is_protected_p1 = (i in protected_indices_p1)
            is_protected_p2 = (i in protected_indices_p2)
            
            if (is_protected_p1 or is_protected_p2) and (gene1 != gene2):
                pass # Block the swap
            else:
                # Standard Swap (Individual genes)
                if random.random() < uniform_crossover_probability:
                    offspring1[i] = gene2
                    offspring2[i] = gene1
        i += 1
                
    # 4. Validation & Return
    offspring_list = []
    for chromosome, original in [(offspring1, p1_chromosome), (offspring2, p2_chromosome)]:
        try:
            temp_dict = {'chromosome': chromosome, 'fitness':0}
            compile_chromosome_to_useable_function(temp_dict, 
                                                dual_bound_functions_dict = dual_bound_functions_registry ,
                                                print_code=False)
            offspring_list.append(chromosome)
        except Exception as e:
            print(f"Uniform crossover produced invalid offspring: {e}")
            print(f"Using fallback parent chromosome for this uniform crossover.")
            offspring_list.append(original)
            
    return offspring_list

## Combined crossover generator

In [102]:
def combined_crossover_generator(parent1_dict, parent2_dict, 
                                dual_bound_functions_registry, 
                                didp_model_registry, 
                                dual_bound_expression_function, 
                                homology_1_point_crossover_probability=homology_1_point_crossover_probability, 
                                subtree_crossover_probability=subtree_crossover_probability,
                                uniform_crossover_probability=uniform_crossover_probability,
                                available_operations=available_operations,
                                reference_point = OPTIMAL_COST_REFERENCE,
                                solver_time_limit = None):
    """
    Randomly selects one of the three crossover methods to produce offspring,
    then evaluates the fitness of the resulting offspring.
    
    Returns:
        list: A list of evaluated dictionaries [{'chromosome': [...], 'fitness': float}, ...]
    """
    crossover_methods = [
        "one_point",
        "subtree",
        "uniform"
    ]
    
    # 1. Select Method
    selected_method = random.choice(crossover_methods)
    
    raw_offspring_chromosomes = []
    
    # 2. Generate Offspring (Raw Lists)
    if selected_method == "one_point":
        # print("Using One-Point Crossover with Homology.")
        raw_offspring_chromosomes = one_point_crossover_two_offspring(
            parent1_dict, parent2_dict, 
            dual_bound_functions_registry = dual_bound_functions_registry, 
            homology_1_point_crossover_probability=homology_1_point_crossover_probability
        )
    
    elif selected_method == "subtree":
        # print("Using Subtree Crossover.")
        single_offspring = subtree_crossover(
            parent1_dict, parent2_dict, 
            dual_bound_functions_registry = dual_bound_functions_registry, 
            subtree_crossover_probability=subtree_crossover_probability
        )
        # Wrap the single result in a list to maintain consistency
        raw_offspring_chromosomes = [single_offspring]
    
    elif selected_method == "uniform":
        # print("Using Uniform Crossover with Weighted Protection.")
        raw_offspring_chromosomes = uniform_crossover_weighted_protected(
            parent1_dict, parent2_dict, 
            dual_bound_functions_registry = dual_bound_functions_registry,  
            uniform_crossover_probability=uniform_crossover_probability,
        )

    # 3. Evaluate Fitness for All Offspring
    evaluated_offspring_list = []
    
    for offspring_chromosome in raw_offspring_chromosomes:
        # Create the dictionary structure
        offspring_chromosome_fitness_dict = {'chromosome': offspring_chromosome, 'fitness': None}
        
        # Calculate fitness
        evaluated_individual = chromosome_fitness_dict_evaluation(
            chromosome_fitness_dict= offspring_chromosome_fitness_dict, 
            didp_model_registry =didp_model_registry, 
            dual_bound_expression_function =dual_bound_expression_function, 
            reference_point = reference_point,
            solver_time_limit = solver_time_limit
        )
        
        evaluated_offspring_list.append(evaluated_individual)
        
    return evaluated_offspring_list

# **8. Mutation operator**

## Subtree mutation

In [110]:
def subtree_mutation(chromosome_fitness_dict, dual_bound_functions_registry, 
                    LB_range_of_constant = LB_range_of_constant, 
                    UB_range_of_constant = UB_range_of_constant, 
                    mutation_max_subtree_depth = mutation_max_subtree_depth, 
                    available_operations = available_operations):
    """
    Performs Subtree Mutation.
    1. Selects a random node (subtree) in the parent chromosome.
    2. Generates a NEW random subtree (RPN list).
    3. Replaces the old subtree with the new one.
    
    Args:
        individual_dict (dict): Parent {'chromosome': [...], ...}
        mutation_max_depth (int): Max depth for the NEWLY generated subtree (usually small, e.g., 2-4).
        [Other args]: Standard generation parameters.
    """
    before_mutated_chromosome = extract_chromosome_from_chromosome_fitness_dict(chromosome_fitness_dict)
    
    # 1. Analyze Structure to find valid cut points
    # We reuse your existing helper to get all valid subtree ranges (Terminals & Functions)
    chromosome_structure = analyzing_chromosome_based_on_rpn_structure(chromosome_fitness_dict = chromosome_fitness_dict, 
                                                                    available_operations = available_operations
                                                                    )
    
    # Combine all possible cut points into one list
    # Each candidate is a tuple (start_index, end_index)
    candidates = chromosome_structure["FUNCTIONS"] + chromosome_structure["TERMINALS"]

    # 2. Select a Mutation Point
    cut_start, cut_end = random.choice(candidates)
    
    # 3. Generate a New Random Subtree
    # We generate a completely new valid RPN expression to graft in.
    # We use 'generate_ramped_half_and_half' to ensure diversity.
    new_subtree = generate_ramped_half_and_half(dual_bound_functions_registry, 
                                                LB_range_of_constant = LB_range_of_constant, 
                                                UB_range_of_constant = UB_range_of_constant,
                                                min_chromosome_length = min_chromosome_length, 
                                                max_chromosome_length=mutation_max_subtree_depth,
                                                available_operations = available_operations
                                                )

    # 4. Grafting (Replace Old with New)
    # Prefix: Everything before the cut
    prefix = before_mutated_chromosome[:cut_start]
    
    # Suffix: Everything after the cut
    suffix = before_mutated_chromosome[cut_end+1:]
    
    # New Chromosome: Prefix + New_Subtree + Suffix
    mutated_chromosome = prefix + new_subtree + suffix
    
    # 5. Validation & Return
    try:
        # Check if valid RPN
        new_individual = {'chromosome': mutated_chromosome, 'fitness': 0}
        compile_chromosome_to_useable_function(chromosome_fitness_dict = new_individual, 
                                            dual_bound_functions_dict = dual_bound_functions_registry, 
                                            print_code=False)
        return mutated_chromosome
    except Exception as e:
        print(f"Subtree mutation produced invalid tree: {e}")
        print(f"Using fallback original chromosome for this subtree mutation instance.")
        return before_mutated_chromosome # Fallback to original

## Point mutation

In [111]:
def point_mutation(chromosome_fitness_dict, dual_bound_functions_registry, 
                LB_range_of_constant = LB_range_of_constant, 
                UB_range_of_constant = UB_range_of_constant, 
                available_operations= available_operations):
    """
    Performs Point Mutation (Bit-Flip equivalent).
    Selects a random node and replaces it with a valid alternative of the same arity.
    
    Constraints enforced:
    1. Weighted Blocks [Num, Str, MUL]: 
        - Num (Coefficient): Can be altered.
        - Str (Heuristic): Can be altered.
        - MUL (Operator): PROTECTED (Cannot be altered).
    2. Alone Coefficients/Heuristics: Can be altered.
    3. Functions: Can be altered (e.g., ADD -> SUBTRACT).
    """
    before_mutation_chromosome = extract_chromosome_from_chromosome_fitness_dict(chromosome_fitness_dict)
    
    # 1. Analyze Structure to identify "Protected" nodes
    # We reuse your existing analyzer to find Weighted Blocks
    chromosome_structure = analyzing_chromosome_based_on_rpn_structure(chromosome_fitness_dict = chromosome_fitness_dict,
                                                                    available_operations = available_operations
                                                                    )
    
    protected_indices = set()
    
    # Identify the 'MULTIPLY' at the end of Weighted Blocks [Num, Str, MUL]
    # The user rule: "if we reached a point belong to a terminal set... dont altered the 'multiply'"
    for start, end in chromosome_structure["TERMINALS"]:
        if (end - start) == 2: # Length 3 block [Num, Str, MUL]
            protected_indices.add(end) # The 'end' index is the protected MULTIPLY
            
    # 2. Identify Valid Mutation Candidates
    # All indices are valid EXCEPT the protected 'MULTIPLY's
    valid_indices = [i for i in range(len(before_mutation_chromosome)) if i not in protected_indices]
    
    if not valid_indices:
        return chromosome_fitness_dict # No mutation possible
        
    # 3. Select Random Node to Mutate
    mutation_idx = random.choice(valid_indices)
    original_gene = before_mutation_chromosome[mutation_idx]
    
    new_gene = original_gene # Default fallback
    
    # 4. Perform Mutation based on Node Type
    
    # --- CASE A: Coefficient (Float/Int) ---
    # "only the coeficients... can be altered"
    if isinstance(original_gene, (int, float)):
        # Generate a new random coefficient
        new_gene = round(random.uniform(LB_range_of_constant, UB_range_of_constant), 2)
        
    # --- CASE B: String (Heuristic or Operator) ---
    elif isinstance(original_gene, str):
        
        # Check if it is an Operator (Arity 2)
        if original_gene in available_operations:
            # "only can altered the function name ('add' -> subtract)"
            # Filter for operators distinct from the original
            candidates = [op for op in available_operations if op != original_gene]
            if candidates:
                new_gene = random.choice(candidates)
                
        # Otherwise, it must be a Heuristic Name (Arity 0)
        else:
            # "only the... dual bound name can be altered"
            # Replace with a different heuristic (e.g., 'h1' -> 'h3')
            dual_bound_names = list(dual_bound_functions_registry.keys())
            candidates = [h for h in dual_bound_names if h != original_gene]
            if candidates:
                new_gene = random.choice(candidates)
    
    # 5. Construct New Chromosome
    mutated_chromosome = before_mutation_chromosome.copy()
    mutated_chromosome[mutation_idx] = new_gene
    
    # 6. Return New Individual
    try:
        # Check if valid RPN
        new_individual = {'chromosome': mutated_chromosome, 'fitness': 0}
        compile_chromosome_to_useable_function(chromosome_fitness_dict = new_individual, 
                                            dual_bound_functions_dict = dual_bound_functions_registry, 
                                            print_code=False)
        return mutated_chromosome
    except Exception as e:
        print(f"Point mutation produced invalid candidates: {e}")
        print(f"Using fallback original chromosome for this point mutation instance.")
        return before_mutation_chromosome

## Combined mutation generator

In [112]:
def combined_mutation_generator(parent_dict, dual_bound_functions_registry, 
                                didp_model_registry, 
                                dual_bound_expression_function,
                                LB_range_of_constant = LB_range_of_constant, 
                                UB_range_of_constant = UB_range_of_constant,
                                mutation_max_subtree_depth = mutation_max_subtree_depth, 
                                available_operations=available_operations,
                                reference_point = OPTIMAL_COST_REFERENCE,
                                solver_time_limit = None):
    """
    Randomly selects one of the mutation methods to produce a mutated offspring,
    then evaluates the fitness of the resulting offspring.
    
    Returns:
        list: A list containing the single evaluated dictionary [{'chromosome': [...], 'fitness': float}]
    """
    mutation_methods = [
        "subtree",
        "point"
    ]
    
    # 1. Select Method
    selected_method = random.choice(mutation_methods)
    
    raw_mutated_chromosome = []
    
    # 2. Generate Mutated Offspring (Raw List or Dict)
    if selected_method == "subtree":
        # print("Using Subtree Mutation.")
        raw_mutated_chromosome = subtree_mutation(
            chromosome_fitness_dict = parent_dict, 
            dual_bound_functions_registry = dual_bound_functions_registry, 
            LB_range_of_constant = LB_range_of_constant, 
            UB_range_of_constant = UB_range_of_constant,
            mutation_max_subtree_depth = mutation_max_subtree_depth, 
            available_operations = available_operations
        )
    
    elif selected_method == "point":
        # print("Using Point Mutation.")
        raw_mutated_chromosome = point_mutation(
            chromosome_fitness_dict = parent_dict, 
            dual_bound_functions_registry = dual_bound_functions_registry, 
            LB_range_of_constant = LB_range_of_constant, 
            UB_range_of_constant = UB_range_of_constant,
            available_operations = available_operations 
        )

    # 3. Normalize Output (Handle cases where mutation returns the original dict vs a new list)
    if isinstance(raw_mutated_chromosome, dict):
        raw_mutated_offspring_chromosome = raw_mutated_chromosome.get('chromosome')
    else:
        raw_mutated_offspring_chromosome = raw_mutated_chromosome

    # 4. Evaluate Fitness
    # Create the dictionary structure
    raw_mutated_chromosome_fitness_dict = {'chromosome': raw_mutated_offspring_chromosome, 'fitness': None}
    
    # Calculate fitness
    evaluated_mutated_chromosome_fitness_dict = chromosome_fitness_dict_evaluation(
        chromosome_fitness_dict = raw_mutated_chromosome_fitness_dict, 
        didp_model_registry = didp_model_registry, 
        dual_bound_expression_function = dual_bound_expression_function, 
        reference_point = reference_point,
        solver_time_limit = solver_time_limit
    )
    
    # Return as a list to match the crossover generator's interface
    return [evaluated_mutated_chromosome_fitness_dict]

# **9. Evolutionary algorithm**

In [114]:
def evolution_algorithm_execution(
    # --- 1. EVOLUTIONARY ALGORITHM HYPERPARAMETERS ---
    population_size=POPULATION_SIZE,
    generations=GENERATIONS,
    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    elitism_rate=ELITISM_RATE, # 1% of best individuals preserved
    
    # --- 2. OPERATOR PARAMETERS ---
    LB_range_of_constant=LB_range_of_constant,
    UB_range_of_constant=UB_range_of_constant,
    min_chromosome_length=min_chromosome_length,
    max_chromosome_length=max_chromosome_length,
    tournament_size=tournament_size,
    tournament_probability=tournament_probability,
    mutation_max_subtree_depth=mutation_max_subtree_depth,
    homology_1_point_crossover_probability=homology_1_point_crossover_probability,
    subtree_crossover_probability=subtree_crossover_probability,
    uniform_crossover_probability=uniform_crossover_probability,
    
    # --- 3. PROBLEM SPECIFIC PARAMETERS ---
    available_operations = available_operations,
    didp_model_registry=None,          
    dual_bound_expression_function=None,
    dual_bound_functions_registry = None,
    
    # --- 4. OTHER PARAMETERS ---    
    reference_point = OPTIMAL_COST_REFERENCE,
    solver_time_limit = None #seconds
    ):
    """
    Executes the full Evolutionary Algorithm lifecycle for Domain-Independent Dynamic Programming.
    """
    # --- TIMING DICTIONARY ---
    timings = {
        "initialization": 0.0,
        "selection": 0.0,
        "crossover": 0.0,
        "mutation": 0.0,
        "logging_overhead": 0.0,
        "total_runtime": 0.0
    }
    
    # --- STEP 0: PREPARE REGISTRY ---
    # We create a static registry ONCE to validate syntax during generation/crossover/mutation.
    # The actual fitness evaluation will create its own fresh registry per model instance.
    overall_start_time = time.time()
    if didp_model_registry is None or dual_bound_expression_function is None:
        raise ValueError("You must provide 'didp_model_registry' and 'dual_bound_expression_function' factories.")
        
    static_model_bundle = didp_model_registry()
    dual_bound_functions_registry = dual_bound_expression_function(static_model_bundle)
    
    print(f"--- Initialization: Generating Population of size {population_size} - {generations} generations - solver limit at {solver_time_limit} seconds ---")
    
    # --- STEP 1: INITIALIZATION ---
    t_start = time.time()
    print("Generating Initial Population at time:", time.ctime())
    population = initialize_list_of_chromosome_fitness_dictionary(
        list_size = population_size, 
        dual_bound_functions_dict = dual_bound_functions_registry,
        didp_model_registry = didp_model_registry, 
        dual_bound_expression_function = dual_bound_expression_function,
        LB_range_of_constant=LB_range_of_constant, 
        UB_range_of_constant=UB_range_of_constant,
        min_chromosome_length=min_chromosome_length, 
        max_chromosome_length = max_chromosome_length, 
        available_operations = available_operations,
        reference_point = reference_point,
        solver_time_limit = solver_time_limit
    )
    timings["initialization"] += time.time() - t_start
    print("Initial Population Generated at time:", time.ctime())
    print("Processing time for initialization:", timings["initialization"])
    
    # Track global best
    best_individual_ever = min(population, 
                            key=lambda x: x['fitness'] if x['fitness'] is not None else float('inf')
                            )
    print(f"Initial Best Fitness: {best_individual_ever['fitness']}")

    # --- STEP 2: GENERATIONAL LOOP ---
    for gen in range(1, generations + 1):
        
        new_population = []
        
        # --- A. REPRODUCTION (Elitism) ---
        # Keep top 1% (or at least 1) best parents directly
        num_elites = max(1, int(population_size * elitism_rate))
        # Sort by fitness (lower is better)
        sorted_pop = sorted(population, 
                            key=lambda x: x['fitness'] if x['fitness'] is not None else float('inf')
                            )
        elites = sorted_pop[:num_elites]
        new_population.extend(elites)
        
        # --- B. OFFSPRING GENERATION ---
        # We need to fill the rest of the population
        while len(new_population) < population_size:
            
            # 1. Selection
            t_start = time.time()
            parent1 = parents_selection(population, tournament_size = tournament_size)
            parent2 = parents_selection(population, tournament_size = tournament_size)
            timings["selection"] += time.time() - t_start
            
            # 2. Crossover
            t_start = time.time()
            if random.random() < crossover_rate:
                # Returns a LIST of evaluated offspring
                offspring_list = combined_crossover_generator(
                    parent1_dict = parent1, 
                    parent2_dict = parent2, 
                    dual_bound_functions_registry = dual_bound_functions_registry,
                    didp_model_registry = didp_model_registry, 
                    dual_bound_expression_function = dual_bound_expression_function,
                    homology_1_point_crossover_probability = homology_1_point_crossover_probability,
                    subtree_crossover_probability = subtree_crossover_probability,
                    uniform_crossover_probability = uniform_crossover_probability,
                    available_operations = available_operations, 
                    reference_point = reference_point,
                    solver_time_limit = solver_time_limit
                )
                
            else:
                # No Crossover: Just clone parents
                # We re-evaluate them just in case (or you could copy fitness)
                offspring_list = [parent1, parent2] 
            timings["crossover"] += time.time() - t_start
            
            # 3. Mutation
            t_start = time.time()
            final_offspring_for_batch = []
            for ind in offspring_list:
                if random.random() < mutation_rate:
                    # Returns a LIST containing the single mutated offspring (evaluated)
                    mutated_list = combined_mutation_generator(
                        parent_dict=ind, 
                        dual_bound_functions_registry=dual_bound_functions_registry,
                        didp_model_registry = didp_model_registry,
                        dual_bound_expression_function = dual_bound_expression_function,
                        LB_range_of_constant = LB_range_of_constant,
                        UB_range_of_constant = UB_range_of_constant,
                        mutation_max_subtree_depth = mutation_max_subtree_depth,
                        available_operations= available_operations,
                        reference_point = reference_point,
                        solver_time_limit = solver_time_limit
                    )
                    final_offspring_for_batch.append(mutated_list[0])
                else:
                    final_offspring_for_batch.append(ind)
            
            # 4. Add to New Population
            for child in final_offspring_for_batch:
                if len(new_population) < population_size:
                    new_population.append(child)
            timings["mutation"] += time.time() - t_start
        
        # --- STEP 3: UPDATE & LOGGING ---
        t_start = time.time()
        population = new_population
        
        # Find best in current generation
        current_best = min(population, 
                        key=lambda x: x['fitness'] if x['fitness'] is not None else float('inf')
                        )
        
        # Update global best
        if (current_best['fitness'] is not None and 
            (best_individual_ever['fitness'] is None or current_best['fitness'] <= best_individual_ever['fitness'])):
            best_individual_ever = current_best
        print(f"Gen {gen}: Best Fitness = {current_best['fitness']} | Global Best = {best_individual_ever['fitness']}")
        timings["logging_overhead"] += time.time() - t_start

    timings["total_runtime"] = time.time() - overall_start_time
    
    # --- PRINT REPORT ---
    print("\n" + "="*40)
    print("       PERFORMANCE PROFILING REPORT       ")
    print("="*40)
    print(f"Total Runtime:    {timings['total_runtime']:.4f} seconds")
    print("-" * 40)
    print(f"Initialization:   {timings['initialization']:.4f}s ({timings['initialization']/timings['total_runtime']*100:.1f}%)")
    print(f"Selection:        {timings['selection']:.4f}s      ({timings['selection']/timings['total_runtime']*100:.1f}%)")
    print(f"Crossover (+Eval):{timings['crossover']:.4f}s      ({timings['crossover']/timings['total_runtime']*100:.1f}%)")
    print(f"Mutation (+Eval): {timings['mutation']:.4f}s       ({timings['mutation']/timings['total_runtime']*100:.1f}%)")
    print(f"Logging/Overhead: {timings['logging_overhead']:.4f}s ({timings['logging_overhead']/timings['total_runtime']*100:.1f}%)")
    print("="*40)


    print("--- Evolution Completed ---")
    print(f"Best Individual Found: {best_individual_ever}")
    return best_individual_ever

# **Execution**

In [115]:
# ==========================================
# 1. EVOLUTIONARY ALGORITHM HYPERPARAMETERS
# ==========================================
POPULATION_SIZE = 2        # Size of the population in each generation
GENERATIONS = 2           # Number of generations to run
MUTATION_RATE = 0.2         # Probability of mutating an individual
CROSSOVER_RATE = 0.8        # Probability of performing crossover
ELITISM_RATE = 0.02
# ==========================================
# 2. OPERATOR PARAMETERS
# ==========================================
# Bounds for the coefficients generated for weighted blocks (e.g., 5.5 * h1)
LB_range_of_constant = 0.0  
UB_range_of_constant = 10.0 
# Depth limits for the RPN trees (used in Ramped Half-and-Half generator)
min_chromosome_length = 2               # Minimum depth of the initial trees
max_chromosome_length = 10               # Maximum depth of the initial trees
# Probability of selecting the best individual in the  tournament selection
# Tournament size for parent selection
tournament_size=random.randint(2, 10)
tournament_probability=0.8
# Mutation: Maximum depth allowed for the *newly generated* subtree during mutation
mutation_max_subtree_depth = random.randint(min_chromosome_length, max_chromosome_length)  # Randomly chosen between 1 and 3
# 1-Point Crossover: Probability of using Homology (matching structure) vs Random fallback
homology_1_point_crossover_probability = 0.5
# Subtree Crossover: Probability of swapping a Function (Branch) vs Terminal (Leaf)
subtree_crossover_probability = 0.9
# Uniform Crossover: Probability of swapping genes at a specific index
uniform_crossover_probability = 0.5
# ==========================================
# 4. OTHER PARAMETERS
# ==========================================
# The Ground Truth optimal cost for the specific problem instance
# Used to calculate fitness (deviation from optimal)
OPTIMAL_COST_REFERENCE= 400
# Time limit (in seconds) for the DIDP solver to run per chromosome evaluation
SOLVER_TIME_LIMIT = 5 #seconds

In [116]:
best_individual_ever = evolution_algorithm_execution(
    # --- 1. EVOLUTIONARY ALGORITHM HYPERPARAMETERS ---
    population_size= POPULATION_SIZE, 
    generations= GENERATIONS, 
    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    elitism_rate=ELITISM_RATE, # 1% of best individuals preserved
    
    # --- 2. OPERATOR PARAMETERS ---
    LB_range_of_constant=LB_range_of_constant,
    UB_range_of_constant=UB_range_of_constant,
    min_chromosome_length=min_chromosome_length,
    max_chromosome_length=max_chromosome_length,
    tournament_size=tournament_size,
    tournament_probability=tournament_probability,
    mutation_max_subtree_depth=mutation_max_subtree_depth,
    homology_1_point_crossover_probability=homology_1_point_crossover_probability,
    subtree_crossover_probability=subtree_crossover_probability,
    uniform_crossover_probability=uniform_crossover_probability,
    
    # --- 3. PROBLEM SPECIFIC PARAMETERS ---
    available_operations = available_operations,
    didp_model_registry=creation_of_didp_model_function,          
    dual_bound_expression_function=dual_bound_expression_function,
    dual_bound_functions_registry = dual_bound_functions_registry,
    
    # --- 4. OTHER PARAMETERS ---    
    reference_point = OPTIMAL_COST_REFERENCE,
    solver_time_limit = SOLVER_TIME_LIMIT #seconds
    )

--- Initialization: Generating Population of size 2 - 2 generations - solver limit at 5 seconds ---
Generating Initial Population at time: Sat Dec 13 13:56:20 2025


Initial Population Generated at time: Sat Dec 13 13:56:30 2025
Processing time for initialization: 10.187968969345093
Initial Best Fitness: 0.0475
Gen 1: Best Fitness = 0.0475 | Global Best = 0.0475
Gen 2: Best Fitness = 0.0475 | Global Best = 0.0475

       PERFORMANCE PROFILING REPORT       
Total Runtime:    20.7916 seconds
----------------------------------------
Initialization:   10.1880s (49.0%)
Selection:        0.0000s      (0.0%)
Crossover (+Eval):5.0611s      (24.3%)
Mutation (+Eval): 5.5001s       (26.5%)
Logging/Overhead: 0.0062s (0.0%)
--- Evolution Completed ---
Best Individual Found: {'chromosome': ['h_1tree', 'h_degree_average', 'MIN', 1.73, 'h_mst', 'MULTIPLY', 7.08, 'h_eigen', 'MULTIPLY', 'SUBTRACT', 'SUBTRACT'], 'fitness': 0.0475}


# **Testing modified DIDP model**

In [91]:
# 1. Setup Chromosome
combined_dual_bound_chromosome = ['h_lp_relaxation']
#[0.79, 'h_1tree', 'MULTIPLY', 'h_degree_average', 5.14, 'h_mst', 'MULTIPLY', 'h_eigen', 'ADD', 'h_assignment', 0.76, 'h_degree_average', 'MULTIPLY', 'h_degree_average', 'SUBTRACT', 'SUBTRACT', 'h_eigen', 8.59, 'h_degree_average', 'MULTIPLY', 'MIN', 'h_1tree', 1.39, 'h_1tree', 'MULTIPLY', 'ADD', 'MIN', 'h_assignment', 'ADD', 'SUBTRACT', 'MAX', 'ADD', 'MIN']
temp_dict = {'chromosome': combined_dual_bound_chromosome, 'fitness':0}

print("The combined dual bounds will be depicted in the following code ")
combined_dual_bound_function = compile_chromosome_to_useable_function(
    temp_dict, 
    dual_bound_functions_dict = dual_bound_functions_registry,
    print_code=True
)

print("\nSuccessfully created solver. Starting search...\n")

# 2. Run Solver
# Ensure 'output_other_result=True' and 'print_timing_stats=True'
result = combining_modified_didppy_solver_with_chromosome(
    combined_dual_bound_chromosome, 
    creation_of_didp_model_function, 
    dual_bound_expression_function, 
    solver_time_limit = 10,
    output_other_result = True,
    print_timing_stats = True
)
display(result)


The combined dual bounds will be depicted in the following code 
Generated Code:
def dual_bound_combination(state):
    return h_lp_relaxation(state)


Successfully created solver. Starting search...


                    📋 SOLVER RUN REPORT                     
🎯 SOLUTION STATUS:
   • Cost:              395
   • Status:            ⚠️  Suboptimal / Timeout
   • Nodes Generated:   1,256
   • Nodes Expanded:    374
   • Branching Factor:  ~3.36
------------------------------------------------------------
⏱️  TIME DISTRIBUTION (Total: 10.0012s):
   [Pure CABS time:   2.9%] 🆚 [Bridge time:  97.1%]

   1. 🟢 Pure CABS (Search):    0.2879 s
      └─ Average time per expanded state:   0.7697 ms
   2. 🔴 Total Bridge Time:     9.7133 s
      ├─ 🐍 Dual Bound Calculation Time: :     9.5969 s  ( 98.8% of bridge time)
      └─ 🌉 Switching Time:  0.1164 s  (  1.2% of bridge time)
------------------------------------------------------------
📊 PYTHON DUAL BOUND CALL STATS:
   • Total Calls:       4,092

None

In [ ]:
# 1. Setup Chromosome
combined_dual_bound_chromosome = [0.36, 'h_assignment', 'MULTIPLY']
temp_dict = {'chromosome': combined_dual_bound_chromosome, 'fitness':0}

print("The combined dual bounds will be depicted in the following code ")
combined_dual_bound_function = compile_chromosome_to_useable_function(
    temp_dict, 
    dual_bound_functions_dict = dual_bound_functions_registry,
    print_code=True
)

print("\nSuccessfully created solver. Starting search...\n")

# 2. Run Solver
# Ensure 'output_other_result=True' and 'print_timing_stats=True'
result = combining_modified_didppy_solver_with_chromosome(
    combined_dual_bound_chromosome, 
    creation_of_didp_model_function, 
    dual_bound_expression_function, 
    solver_time_limit = 100,
    output_other_result = True,
    print_timing_stats = True
)
display(result)


In [ ]:
"""
The combined dual bounds will be depicted in the following code 
Generated Code:
def dual_bound_combination(state):
    return (0.36 * h_assignment(state))


Successfully created solver. Starting search...


============================================================
                    📋 SOLVER RUN REPORT                     
============================================================
🎯 SOLUTION STATUS:
   • Cost:              400
   • Status:            ⚠️  Suboptimal / Timeout
   • Nodes Generated:   7,268,726
   • Nodes Expanded:    3,622,655
   • Branching Factor:  ~2.01
------------------------------------------------------------
⏱️  TIME DISTRIBUTION (Total: 2951.9228s):
   [Pure CABS time:  24.4%] 🆚 [Bridge time:  75.6%]

   1. 🟢 Pure CABS (Search):    720.5010 s
      └─ Average time per expanded state:   0.1989 ms
   2. 🔴 Total Bridge Time:     2231.4218 s
      ├─ 🐍 Dual Bound Calculation Time: :     2010.6213 s  ( 90.1% of bridge time)
      └─ 🌉 Switching Time:  220.8005 s  (  9.9% of bridge time)
------------------------------------------------------------
📊 PYTHON DUAL BOUND CALL STATS:
   • Total Calls:       34,890,005
   • Avg Bridge Time:   0.0640 ms
   • Avg Pure Calc:     0.0576 ms
   • Avg Switching:     0.0063 ms
   • Throughput:        11,819 calls/sec
============================================================
"""

# **Testing original DIDP model**

In [32]:
# 1. Parse Data
n = number_of_customers
c = distance_list

# 2. Initialize Model
# Note: Ensure float_cost matches your data. Your snippet used False (Int), 
# so we explicitly cast distances to Int in the reader.
model = m_dp.Model(maximize=False, float_cost=False)

customer = model.add_object_type(number=n)

# 3. State Variables
# U: Unvisited set (excluding depot 0)
unvisited = model.add_set_var(object_type=customer, target=list(range(1, n)))
# i: Current location
location = model.add_element_var(object_type=customer, target=0)

# 4. Resource Tables
travel_time = model.add_int_table(c)

# 5. Transitions
# Visit customer j
for j in range(1, n):
    visit = m_dp.Transition(
        name="visit {}".format(j),
        cost=travel_time[location, j] + m_dp.IntExpr.state_cost(),
        preconditions=[unvisited.contains(j)],
        effects=[
            (unvisited, unvisited.remove(j)),
            (location, j),
        ],
    )
    model.add_transition(visit)

# Return to depot
# Note: Removed 'time' effect from your snippet as it wasn't defined in the variables
return_to_depot = m_dp.Transition(
    name="return",
    cost=travel_time[location, 0] + m_dp.IntExpr.state_cost(),
    effects=[
        (location, 0),
    ],
    preconditions=[unvisited.is_empty(), location != 0],
)
model.add_transition(return_to_depot)

# 6. Base Case
model.add_base_case([unvisited.is_empty(), location == 0])

# 7. Dual Bounds (from your snippet)
# Min outgoing edge for remaining nodes
min_to = model.add_int_table(
    [min(c[k][j] for k in range(n) if k != j) for j in range(n)]
)
model.add_dual_bound(min_to[unvisited] + (location != 0).if_then_else(min_to[0], 0))

# Min incoming edge for remaining nodes
min_from = model.add_int_table(
    [min(c[j][k] for k in range(n) if k != j) for j in range(n)]
)
model.add_dual_bound(
    min_from[unvisited] + (location != 0).if_then_else(min_from[location], 0)
)

t_start = time.time()
solver = m_dp.CABS(
        model,
        quiet=False,
        time_limit = 10 #None
    )

print("Successfully created solver. Starting search...")
solution = solver.search()
print("Transitions to apply:")
print("")
for t in solution.transitions:
    print(t.name)
print("")
print("Cost: {}".format(solution.cost))
t_end = time.time()
duration = t_end - t_start
print(f"Total solving time is {duration} seconds")
print('Solution optimality: {}'.format(solution.is_optimal))
if not solution.is_optimal:
  print(f"Optimality is not reached yet")
print("Number of nodes generated: {}".format(solution.generated))
print("Number of nodes expanded: {}".format(solution.expanded))

Successfully created solver. Starting search...
Transitions to apply:

visit 18
visit 19
visit 12
visit 6
visit 16
visit 7
visit 4
visit 17
visit 8
visit 14
visit 10
visit 2
visit 15
visit 1
visit 11
visit 13
visit 3
visit 5
visit 9
return

Cost: 400
Total solving time is 10.03556752204895 seconds
Solution optimality: False
Optimality is not reached yet
Number of nodes generated: 380826
Number of nodes expanded: 156908


In [ ]:
"""Successfully created solver. Starting search...
Transitions to apply:

visit 18
visit 19
visit 12
visit 6
visit 16
visit 7
visit 4
visit 17
visit 8
visit 14
visit 10
visit 2
visit 15
visit 1
visit 11
visit 13
visit 3
visit 5
visit 9
return

Cost: 400
Total solving time is 180.40878748893738 seconds
Solution optimality: True
Number of nodes generated: 4104403
Number of nodes expanded: 3346822"""